
# Notebook 10 -- gate B and C

**Asks:** apply the two registered gate clauses and say which facets pass.

**Reads** `ensemble_scores.csv` and `inter_model_icc.csv` (notebook 07),
`facet_dictionary_scores.csv` (notebook 01), `word_list_coverage.csv`
(notebook 02), `facet_similarity.csv` (notebook 09), `retribution_scored.csv`
for the survey scales, and -- only when `HUMAN_GOLD` is True --
`human_match_results.csv` from notebook 99.

**Writes** `gate_results.csv` (one row per facet, every clause and every number
behind it), `stream_agreement_matrix.csv` (how far the three scorers agree, facet
by item), and `gate_c_facet_by_scale_matrix.csv` (the full facet x scale
correlation matrix clause C is decided on).

### The gate, in the registration's own words

Prereg 6.7: "A facet supports a claim only if the models agree at an intraclass
correlation of at least .60 and the facet correlates at least .30 with its
matching survey scale." Two clauses, and this notebook applies both.

- **Clause B, reliability.** Do the models agree with each other? An intraclass
  correlation -- the share of the variation in the scores that is real differences
  between responses rather than differences between raters -- of at least .60,
  **and** no shared-error flag on the facet (below).
- **Clause C, construct validity.** Does the facet track the survey scale it is
  supposed to track? A correlation of at least .30 with its matching scale, with
  word count partialled out and facet scores standardized within vignette.

**Clause A, agreement with human coders, is this design's addition and not the
registration's.** It runs only when `HUMAN_GOLD` is True, and everything about it
lives in notebook 99. Where it did not run, the table below says `not run`. It
never says nothing, and it never says fail.

**The shared-error cap on clause B is the same kind of addition.** An intraclass
correlation cannot tell agreement from correlated error: three models drawn from
overlapping training corpora can be reliably wrong in the same direction. Notebook
99 flags a facet whose ensemble is systematically biased against the human coders
or whose models err on the same responses in the same direction, and a flagged
facet is capped here however high its correlation. With `HUMAN_GOLD` False there
is no flag to read, nothing is capped, and the notebook says so.

**Every file this notebook reads or writes is checked to be one row per key.**
The corpus is one row per participant per item; nothing upstream of notebook 00
guarantees that, so it is checked here on the way in and again on the way out.

### What passing means, and what it does not

Prereg 2.4 puts the whole of E3 to E7 in one exploratory family. What this gate
decides is a facet's STANDING inside that family -- **gate-backed (primary)** or
**descriptive-only** -- and nothing it can return makes a language test
confirmatory.

Prereg 5.15 names exactly four facet-to-scale mappings: desert, proportionality,
revenge and suffering. Censure and moral balance have no anchor there and the
survey instrument has no items for them, so clause C has nothing admissible to
test them against. **A facet with no anchor cannot be gated on clause C, and that
is consistent with the registration rather than a failure of the facet.** The
reason column says which of the two it is for every row.


## Config


In [ ]:

# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "10"                     # this notebook's number
NB_NAME     = "gate_B_and_C"           # this notebook's job, in two words
NB_ASKS     = "Apply the two registered gate clauses and say which facets pass."


## Setup


In [ ]:

# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")


In [ ]:

# === SETUP (cont.): imports and display width ===
# WHAT THIS CELL DOES: imports the three libraries this notebook needs and widens
# the printed tables. It computes nothing.
#
# NOTHING IS INSTALLED HERE. Every number below comes out of pandas, numpy and
# scipy, all three of which ship with the Colab image. The two intraclass
# correlations are written out from their definition further down and checked
# against the value notebook 07 computed with pingouin, so two independent routes
# to the same quantity have to agree before any of it is believed.
#
# WHY THE DISPLAY WIDTH MATTERS: the gate table is wide (facet x clause x estimate
# x interval). At the default width pandas replaces the middle columns with an
# ellipsis, and a truncated gate table is worse than no gate table -- a verdict
# that is not on the screen still reads as if it had been checked.
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr, t as t_dist
import warnings

# WHAT IS SILENCED, AND WHAT IS DELIBERATELY LEFT AUDIBLE.
# Only RuntimeWarning is hidden. Those are the notices numpy raises on a degenerate
# slice -- the mean of an all-missing column, a division by a zero standard
# deviation -- which this notebook produces by the hundred on the sparse facets and
# which are answered by the n printed beside every estimate. Silencing EVERY
# warning instead would hide the ones worth reading: a deprecation notice, a
# SettingWithCopy notice, a statsmodels ill-conditioning note.
#
# THE ONE EXEMPTION, AND WHY IT NEEDS A SECOND LINE. scipy's constant-input and
# near-constant-input notices are RuntimeWarning SUBCLASSES, so the line above
# would swallow them too -- and they are the most informative warning this
# notebook can raise. One of them on a word-list correlation says the word list
# fired on so few responses that the corroborating leg is uninformative, which is
# exactly what a reader has to know about E4's preregistered corroborator. The
# second line re-enables that family. Filters are consulted newest first, so it
# takes precedence over the blanket line above.
#
# It is a backstop, not the main report. leg_r() below refuses a zero-variance
# column itself and returns no number, and the agreement matrix says so in words
# (dict_leg = 'below floor', ens_dict_r = '--'), so the ordinary sparse case is
# reported rather than warned about. This line is what keeps the warning audible
# in the cases those guards do not cover.
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from scipy.stats import DegenerateDataWarning
    warnings.filterwarnings("always", category=DegenerateDataWarning)
except ImportError:
    # Older scipy has no such class. Nothing is broken; the notice simply arrives
    # as a plain RuntimeWarning and is silenced, so say so rather than assume.
    print("NOTE: this scipy is too old to name its degenerate-input warnings, so")
    print("      they cannot be exempted. Read the n beside every estimate.")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
np.set_printoptions(precision=4, suppress=True)

print("Imports ready: pandas, numpy, scipy. Nothing to install for this notebook.")


In [ ]:

# === SETUP (cont.): the facet vocabulary and the three items ===
# The nine facets in prereg 5.15's order: four principled-retributive (desert,
# proportionality, censure, moral_balance), two for retribution's harder edge
# (revenge, suffering), then the three consequentialist goals, which are scored so
# that retributive language has a full menu of punishment justifications to be
# measured against and are NEVER summed into any retributivism total.
FACETS = ["desert", "proportionality", "censure", "moral_balance",
          "revenge", "suffering", "deterrence", "incapacitation", "rehabilitation"]

# ranking = Q333, the goal-ranking explanation; backward = Sentence_OpenEnd1, the
# sentence justification; forward = Sentence_OpenEnd2, the accomplishment item.
# The order is fixed here so every printed table reads left to right the same way.
ITEM_TYPES = ["ranking", "backward", "forward"]

# The class groupings, carried from the dictionary and the ensemble. PRINCIPLED and
# VENGEFUL are the two sides of E4's co-occurrence question; CONSEQUENTIALIST is a
# third axis and never part of a retributive total.
FACET_CLASS = {
    "desert": "principled", "proportionality": "principled",
    "censure": "principled", "moral_balance": "principled",
    "revenge": "vengeful", "suffering": "vengeful",
    "deterrence": "consequentialist", "incapacitation": "consequentialist",
    "rehabilitation": "consequentialist",
}

# RUN_MODE = "smoke" is a plumbing check, not a measurement. Notebook 00 trims the
# corpus to a handful of PARTICIPANTS, and this notebook simply consumes whatever it
# finds; it does no trimming of its own.
#
# READ NO VERDICT OFF A SMOKE RUN. The clauses will still print pass and fail on a
# twenty-row folder -- an intraclass correlation and a partial correlation are both
# computable on that many rows, they are just meaningless there, and the failure
# mode is that they look exactly like real verdicts. Nothing below marks them. The
# mode is stamped into the manifest by done(), so a smoke table can be identified
# later, but on the screen it is indistinguishable from the real thing.
if RUN_MODE == "smoke":
    print("RUN_MODE = 'smoke': the clause verdicts below prove the plumbing works.")
    print("They are NOT results, and they will NOT look different from results:")
    print("pass and fail will both appear. Re-run in 'full' before reading any of it.")

print(f"Facets: {len(FACETS)}. Items: {', '.join(ITEM_TYPES)}.")


## Load


In [ ]:

# === LOAD (1 of 3): the three files the gate cannot run without ===
# WHAT THIS CELL DOES: fetches the required inputs through need(), which either
# hands back a path or stops with a sentence saying which notebook makes the file.
#
# CONSUMES
#   ensemble_scores.csv          notebook 07. Long: one row per response x item x
#                                facet, carrying the ensemble mean AND each model's
#                                own score in a score_<model> column. The per-model
#                                columns are what clause B's confidence interval is
#                                resampled from; the mean is what clause C
#                                correlates against the survey.
#   inter_model_icc.csv          notebook 07. One row per facet: the ICC point
#                                estimate, the number of raters behind it, and 07's
#                                own clause B reading of that point estimate. This
#                                notebook adds the interval and decides on its lower
#                                bound.
#   facet_dictionary_scores.csv  notebook 01. The word-list stream, and the only
#                                place n_words and vignette travel from. Clause C
#                                partials word count out and standardizes within
#                                vignette, so both are load-bearing here.
ENS_PATH  = need("ensemble_scores.csv", produced_by="07")
ICC_PATH  = need("inter_model_icc.csv", produced_by="07")
DICT_PATH = need("facet_dictionary_scores.csv", produced_by="01")

ens     = pd.read_csv(ENS_PATH)
icc_in  = pd.read_csv(ICC_PATH)
dict_df = pd.read_csv(DICT_PATH)

# The roster of raters is read off the columns that are actually there rather than
# named here. A run in which one model never scored would otherwise report a
# three-model reliability computed from two, with the printout still saying three.
MODELS = [c[len("score_"):] for c in ens.columns if c.startswith("score_")]

# The separate-call E4 rows (facet = 'desert_solo' and the like) ride in the same
# file and are NOT gate targets: the gate is applied to the joint-call score of
# record. Filtering them out here rather than later keeps them from turning up as
# two extra rows in the gate table.
_extra = sorted(set(ens["facet"].unique()) - set(FACETS))
ens = ens[ens["facet"].isin(FACETS)].copy()

print(f"  ensemble rows:   {len(ens):,}  ({ens.response_id.nunique():,} responses)")
print(f"  raters:          {len(MODELS)}  ({', '.join(MODELS) if MODELS else 'none'})")
if _extra:
    print(f"  set aside, not gate targets: {', '.join(_extra)}")
print(f"  clause B input:  {len(icc_in)} facet rows from notebook 07")
print(f"  dictionary rows: {len(dict_df):,}")


In [ ]:

# === LOAD (2 of 3): the optional inputs, and what each absence costs ===
# NONE of these stops the notebook. Each absence prints one line, changes exactly
# one thing in the output, and is reported in the output as "not run" rather than
# left blank. A blank cell in a gate table cannot be told apart from a failure, and
# the two mean opposite things: one says the clause was applied and the facet did
# not clear it, the other says nobody applied the clause.
#
#   retribution_scored.csv  the survey scaffold from the R pipeline, N = 948 after
#                           the preregistered 6.2 exclusions. Clause C correlates
#                           each facet against a SURVEY scale, so without this file
#                           clause C cannot run at all.
#   word_list_coverage.csv  notebook 02. Says which facets the word list fires on
#                           often enough to corroborate anything (the 5% floor).
#                           Without it the dictionary leg of the agreement matrix
#                           is reported as not run -- it is not recomputed here,
#                           because a rule applied in two places drifts apart.
#   facet_similarity.csv    notebook 09. The embedding leg. It gates NOTHING: no
#                           clause reads it, and the registered gate is inter-model
#                           ICC plus the facet-to-survey correlation.
#   human_match_results.csv notebook 99, the optional human-coding arm, and only
#                           when HUMAN_GOLD is True. It carries clause A and the
#                           shared-error diagnostic. Clause A is this design's
#                           addition to the registration, not part of it.
SURV_PATH = need("retribution_scored.csv", produced_by="00", optional=True)
COV_PATH  = need("word_list_coverage.csv", produced_by="02", optional=True)
SIM_PATH  = need("facet_similarity.csv", produced_by="09", optional=True)
GOLD_A_PATH = (need("human_match_results.csv",
                    produced_by="99, the optional human-coding notebook",
                    optional=True)
               if HUMAN_GOLD else None)
if not HUMAN_GOLD:
    print("  [skip] clause A: HUMAN_GOLD is False, so the human arm never runs. The")
    print("         gate applied below is the two-clause gate of prereg 6.7.")

survey = pd.read_csv(SURV_PATH) if SURV_PATH else None
if survey is not None:
    # The participant id column in the R export is ResponseId. Renamed rather than
    # merged on two names, so the join key is one column everywhere in the chain.
    if "ResponseId" in survey.columns and "response_id" not in survey.columns:
        survey = survey.rename(columns={"ResponseId": "response_id"})
    # THE LITERAL REVENGE SCALE (design section 0, the revenge-scale row). The
    # registered revenge scale averages revenge_1 to _4, and revenge_4 ("Rather than
    # showing criminals so much tolerance, we should be punishing them more
    # severely") asks for harsher punishment without any reference to getting
    # even; it correlates .78 with punitiveness. Every clause C comparison that uses
    # the revenge scale is therefore run a second time on the two literal items --
    # "Society has the right to take revenge on criminal offenders" and "Society
    # should punish to get back at criminal offenders" -- and a facet must pass BOTH
    # (DO 13). Scored with the at-least-half rule every composite uses (one of two
    # items). The registered `revenge` column is left exactly as the file has it.
    if {"revenge_1", "revenge_2"}.issubset(survey.columns):
        _lit = survey[["revenge_1", "revenge_2"]].apply(pd.to_numeric, errors="coerce")
        survey["revenge_lit"] = _lit.mean(axis=1).where(_lit.notna().sum(axis=1) >= 1)
        # GUARD: the registered column must be the four-item mean this comparison
        # assumes. If the export's `revenge` were ever built from other items, the
        # two scorings below would silently compare two other things.
        if {"revenge", "revenge_3", "revenge_4"}.issubset(survey.columns):
            _four = survey[[f"revenge_{_k}" for _k in range(1, 5)]].apply(
                pd.to_numeric, errors="coerce")
            _four_mean = _four.mean(axis=1).where(_four.notna().sum(axis=1) >= 2)
            assert np.allclose(_four_mean.to_numpy(float),
                               pd.to_numeric(survey["revenge"], errors="coerce").to_numpy(float),
                               equal_nan=True), (
                "retribution_scored.csv's `revenge` is not the mean of revenge_1 to _4, "
                "so the literal-scale comparison cannot be read against it.")
        print("  literal revenge scale built from revenge_1 and revenge_2.")
    else:
        print("  NOTE: revenge_1 / revenge_2 are not in the survey file, so the literal")
        print("  revenge scale cannot be built; clause C on it is recorded as not run.")
    print(f"  survey scaffold: {len(survey)} participants (expect 948).")
else:
    print("  No retribution_scored.csv: clause C will be recorded as NOT RUN for")
    print("  every facet. That is a missing input, not a failed gate.")

coverage = pd.read_csv(COV_PATH) if COV_PATH else None
sim = pd.read_csv(SIM_PATH) if SIM_PATH else None
clauseA_in = pd.read_csv(GOLD_A_PATH) if GOLD_A_PATH else None


In [ ]:

# === LOAD (3 of 3): the one-row-per-participant contract, ENFORCED HERE ===
# THE CONTRACT. The corpus is 948 participants x 3 open-ended items = 2,844 rows,
# keyed on (response_id, item_type). Notebook 00 checks it once, for
# openends_long.csv. It is checked HERE for every file this notebook reads, and
# again before every file this notebook writes. Nothing upstream of notebook 00
# guarantees it, and no comment in this notebook claims that anything does.
#
# WHAT BREAKS WITHOUT IT. A duplicated key does not announce itself. The pivot in
# DO (6) would silently AVERAGE two copies of a response rather than raise; a merge
# would fan one row into two; and every n computed below would grow while the
# printed corpus size stayed at 948. Appending three duplicated rows to
# facet_dictionary_scores.csv moves the downstream sample sizes off 948 without one
# line of output changing.
#
# TWO KINDS OF FILE, TWO KINDS OF RESPONSE. A REQUIRED file stops the notebook: the
# gate cannot be run around a corrupt input it depends on. An OPTIONAL file
# DEGRADES: it costs the one analysis that reads it and nothing else, because
# taking the registered gate down over a defect in a file it does not need would be
# the worse error.
def _dupes(frame, keys):
    """(count of rows sharing a key, one example key) or (0, '')."""
    d = frame.duplicated(keys, keep=False)
    if not d.any():
        return 0, ""
    return int(d.sum()), str(frame.loc[d, keys].iloc[0].to_dict())

# The join key is compared as TEXT on every side. Without this, a character
# ResponseId from the R export meeting an integer id from anywhere else raises
# pandas' "trying to merge on object and int64 columns" -- a hard stop with a
# message no reader can act on, which also pre-empts the id-space warning further
# down that was written for exactly that fault.
for _f in (ens, dict_df):
    _f["response_id"] = _f["response_id"].astype(str)
if survey is not None:
    survey["response_id"] = survey["response_id"].astype(str)
if sim is not None and "response_id" in sim.columns:
    sim["response_id"] = sim["response_id"].astype(str)

_n, _ex = _dupes(ens, ["response_id", "item_type", "facet"])
assert _n == 0, (f"ensemble_scores.csv is not unique on (response_id, item_type, "
                 f"facet): {_n} rows share a key, e.g. {_ex}. The pivot below "
                 "would average the copies rather than fail. Re-run notebook 07.")
_n, _ex = _dupes(dict_df, ["response_id", "item_type"])
assert _n == 0, (f"facet_dictionary_scores.csv is not unique on (response_id, "
                 f"item_type): {_n} rows share a key, e.g. {_ex}. Re-run 01.")
assert not icc_in.duplicated(["facet"]).any(), (
    "inter_model_icc.csv must hold one row per facet. This notebook reads each "
    "facet's row with .iloc[0], so a second row would be discarded unread.")
print("  contract: ensemble, word list and ICC files are unique on their keys.")


In [ ]:

# === LOAD (3 of 3, cont.): the same contract on the optional files ===
# Each of these DEGRADES rather than stops. The variable is set to None, the
# analysis that reads it is recorded as not run further down, and the notebook
# carries on.
SURVEY_NOT_RUN = ""      # why clause C could not run, in words, or "" if it can
if survey is not None:
    _n, _ex = _dupes(survey, ["response_id"])
    if _n:
        SURVEY_NOT_RUN = (f"retribution_scored.csv holds {_n} rows sharing a "
                          "participant id")
        print(f"  STOP AND READ: {SURVEY_NOT_RUN}, e.g. {_ex}.")
        print("  One survey row fans out onto that person's three item rows, so a")
        print("  duplicate would double a participant and every clause C")
        print("  correlation would be computed on doubled rows. Clause C is")
        print("  recorded as NOT RUN. Clauses A and B are unaffected and still run.")
        survey = None

# SURVEY_COLS is the record of which names came off the survey file. Clause C
# checks every scale it is about to correlate against this set, because six of the
# nine facet names are also survey scale names.
SURVEY_COLS = set(survey.columns) if survey is not None else set()
if survey is None and not SURVEY_NOT_RUN:
    SURVEY_NOT_RUN = "retribution_scored.csv is not in the folder"

if coverage is not None:
    # cov_lookup() reads one row per (facet, scope) with .iloc[0]; a duplicate
    # would silently pick one of two coverage verdicts.
    _n, _ex = _dupes(coverage, ["facet", "scope"])
    if _n:
        print(f"  word_list_coverage.csv holds {_n} rows sharing a (facet, scope) "
              f"key, e.g. {_ex}. The word-list leg of the agreement matrix is")
        print("  recorded as not run rather than decided by whichever row sorted")
        print("  first. Re-run notebook 02. No clause reads this file.")
        coverage = None

if sim is not None:
    _keys = [c for c in ["response_id", "item_type", "facet"] if c in sim.columns]
    _n, _ex = _dupes(sim, _keys) if len(_keys) == 3 else (0, "")
    if _n:
        print(f"  facet_similarity.csv holds {_n} rows sharing a (response_id, "
              f"item_type, facet) key, e.g. {_ex}. The merge in the agreement")
        print("  matrix would fan one ensemble score onto two similarities. The")
        print("  embedding leg is recorded as not run. It gates nothing.")
        sim = None

print("  contract: optional files checked; a duplicate costs its own leg only.")


## Do


In [ ]:

# === DO (1 of 15): the numbers that decide the clauses ===
# WHAT THIS CELL DOES: fixes the thresholds. Nothing is computed here. Everything
# below reads these constants, so this is the one place to look when asking what
# bar a facet was held to.

ICC_FLOOR   = 0.60   # PREREGISTERED. Prereg 6.7: "the models agree at an intraclass
                     # correlation of at least .60". Not an implementer's choice.
R_OWN_FLOOR = 0.30   # PREREGISTERED. Prereg 6.7: "the facet correlates at least .30
                     # with its matching survey scale".

# ADDITIONS, carried on the dated OSF addendum (design section 0). Both can only
# DEMOTE a facet, never promote one: a facet that fails the registered floor cannot
# be rescued by either of them.
DELTA_R_FLOOR   = 0.15   # the diagonal must clear its most confusable rival by this
                         # much. .15 is roughly the smallest gap that is legible at
                         # this sample size once the intervals are also required not
                         # to overlap -- the interval test, not this constant, is
                         # what does most of the work.
COLLINEAR_FLOOR = 0.60   # when the facet's own scale and the rival scale are
                         # themselves near-copies of each other, "beating the rival"
                         # is a statement about the survey, not about the facet. The
                         # facet is demoted rather than credited.

# WHY THE INTERVAL'S LOWER BOUND AND NOT THE POINT ESTIMATE
# This is the second registered deviation operating in this notebook. The prereg
# sets its thresholds on the estimate; this gate sets them on the lower bound of the
# bootstrap 95% interval. At the base rates these facets carry -- revenge fires on
# under 1% of responses, suffering on about 2.5% -- a coefficient computed on a
# handful of positive cases has a 95% half-width around .26, so a facet whose true
# value is .40 can post a sample .60 on luck alone. Gating on the bound is what
# those base rates require. The preregistered quantity, the point estimate, is
# reported next to every decision, and so is the width of the interval, so a
# wide-but-lucky pass is visible on the face of the table.

# THE ROUTED GATE SITE. The gate is evaluated on the backward item -- Q1,
# "please explain why you recommended this sentence" -- because it is the cleanest
# register match to the closed-ended retributivism scales (design section 3).
# Gating on the ranking explanation instead would test the facets against a
# different question from the one the survey scales ask.
GATE_ITEM = "backward"

# PRESENCE, WHERE A FACET HAS TO BE COUNTED (design section 0, the presence row).
# Every clause and every E3-E7 test runs on the continuous ensemble mean. The two
# counts this notebook relies on use definitions fixed before scoring:
#   the positive-case floor on clause B counts TRACE presence (any model above
#     zero); notebook 07 counts it and DO (9) applies it to the verdict;
#   proportionality's base rate (DO 15) counts CLEAR presence, the ensemble mean
#     above PRESENT_CLEAR.
# DANGER: notebook 07 holds the same constant, and DO (9) stops if the value 07
# wrote into inter_model_icc.csv differs from this one.
PRESENT_CLEAR = 0.50

print(f"Registered floors: ICC >= {ICC_FLOOR:.2f} (clause B), "
      f"r >= {R_OWN_FLOOR:.2f} (clause C).")
print(f"Decided on the 95% interval's LOWER BOUND, at the {GATE_ITEM} item.")


In [ ]:

# === DO (2 of 15): which facets the gate can be applied to at all ===
# THE TWO ROUTINGS BELOW ARE NOT THE SAME KIND OF DECISION. Keep them apart when
# writing this up; the reason column of gate_results.csv keeps them apart on disk.
#
#   CENSURE and MORAL_BALANCE -- CONSISTENT WITH THE PREREG, no amendment needed.
#     Prereg 5.15 names exactly four facet-to-scale mappings (desert with the
#     just-deserts purpose items, proportionality with the proportionality item,
#     revenge with the revenge items, suffering with the suffer items) and names
#     none for censure or moral balance. The instrument simply has no censure or
#     moral-balance items, so clause C has no admissible anchor to test against.
#     The available proxies each correlate .48 to .80 with `deserved`, so pressing
#     one into service would launder the desert scale under a new name and
#     manufacture the very convergence the clause exists to test. These two are
#     reported as prevalence and reliability only. A facet with no anchor CANNOT be
#     gated on clause C, and that is the registration's position, not a failure.
#
#   PROPORTIONALITY -- A REGISTERED DEVIATION, and it goes on the dated addendum.
#     Prereg 5.15 DOES anchor proportionality and prereg 6.7 DOES gate it like the
#     other three. Routing it to descriptive-only is therefore a departure from the
#     registration, not an implication of it. The reason is base rate:
#     proportionality language appears in roughly 1% of responses (0.3% on the
#     backward item), which leaves near-zero predictor variance -- no stable r >= .30
#     is estimable, and there are no cases at all in which to observe the desert = 1
#     / proportionality = 0 dissociation that E5 is built to detect. The decision was
#     made in advance, and the preregistered proportionality gate numbers are
#     computed and reported here anyway, so the demotion is auditable rather than
#     assumed.
#
#   The three CONSEQUENTIALIST goals are scored so retributive language has
#     something to be compared against (prereg 5.15). They are not gate targets and
#     no retributive claim routes through them.

GATE_FACETS = ["desert", "revenge", "suffering"]   # the gate can promote these
# Gate numbers computed and printed, standing fixed in advance regardless of them.
REPORT_ONLY_FACETS = ["proportionality"]
FORCED_DESCRIPTIVE_ONLY = ["censure", "moral_balance", "proportionality"]
CONSEQUENTIALIST = ["deterrence", "incapacitation", "rehabilitation"]

# EVERY FACET MUST BE ROUTED BY ONE OF THE THREE LISTS ABOVE, and the routing cell
# below reads all three by name rather than letting anything fall through to a
# default. This assertion is what makes that true rather than intended: a tenth
# facet added to FACETS and to neither list would otherwise arrive in the gate
# table wearing the standing of whichever branch happened to catch it, with a
# reason line describing a facet it is not.
_unrouted = [f for f in FACETS if f not in
             set(GATE_FACETS) | set(FORCED_DESCRIPTIVE_ONLY) | set(CONSEQUENTIALIST)]
assert not _unrouted, (
    f"these facets are in FACETS but in none of the three routing lists: "
    f"{_unrouted}. Add each one to GATE_FACETS, FORCED_DESCRIPTIVE_ONLY or "
    "CONSEQUENTIALIST and say in the comment above which kind of decision it is.")

print("Gate-eligible (a pass here earns primary standing):", ", ".join(GATE_FACETS))
print("Descriptive-only whatever the numbers say:", ", ".join(FORCED_DESCRIPTIVE_ONLY))
print("Convergent sanity checks only:", ", ".join(CONSEQUENTIALIST))
print("Gate-backed is NOT confirmatory: prereg 2.4 puts all of E3-E7 in one")
print("exploratory family, and the gate decides standing inside it, nothing more.")


In [ ]:

# === DO (3 of 15): each facet's own scale, and the rival it must beat ===
# These come straight from prereg 5.15, which names four facet-to-scale mappings:
# desert with the just-deserts purpose items (Retribution_1 to _4), proportionality
# with the proportionality item, revenge with the revenge items, suffering with the
# suffer items. There is no fifth mapping to be had.
VALIDATOR = {"desert": "deserved", "revenge": "revenge", "suffering": "suffering",
             "proportionality": "proportionality_jd"}

# The candidate scales in the discriminant matrix. The `retribution` composite is
# excluded because it correlates .968 with `deserved` -- a rival that is a near-copy
# of the diagonal makes the margin meaningless in both directions. proportionality_jd
# stays in as the near-orthogonal reference (r = .05 with `deserved`): it is the
# prereg's "unrelated scale", computed and reported, but it is not what decides.
# revenge_lit, the two literal revenge items, is carried as a candidate scale so
# the exported facet-by-scale matrix shows every facet against it. It enters no
# registered verdict: VALIDATOR and RIVALS name only the registered scales, and the
# literal-scale verdict is computed separately in DO (13) (design section 0).
SURVEY_SCALES = ["deserved", "revenge", "revenge_lit", "suffering", "proportionality_jd"]

# THE RIVAL EACH FACET MUST BEAT, and why it is the confusable one rather than the
# unrelated one. Prereg 6.7 asks that "each facet should relate more strongly to its
# own scale than to an unrelated one". Taken literally that is the easiest test in
# the matrix: proportionality_jd is near-orthogonal to everything, so a scorer that
# collapsed every kind of punitive sentiment into one undifferentiated "this offender
# should be punished" reading would pass it while being completely blind to the
# desert-versus-revenge distinction this study exists to test. Replacing the
# unrelated foil with the maximally confusable one is a registered deviation, and
# the preregistered comparison is computed and reported beside it.
#   desert must out-correlate BOTH harder-edge scales -- that is the study's central
#     distinction, and if desert text cannot beat the revenge scale then that is the
#     real ceiling on the principled-versus-vengeful reading, surfaced rather than
#     defined away.
#   revenge and suffering must each out-correlate `deserved`, the scale they are
#     most likely to be mistaken for by a scorer that hears all punishment talk as
#     one thing.
RIVALS = {"desert": ["revenge", "suffering"],
          "revenge": ["deserved"],
          "suffering": ["deserved"],
          "proportionality": ["deserved"]}

print("Facet -> its preregistered survey validator:")
for _f, _s in VALIDATOR.items():
    _tag = "gated" if _f in GATE_FACETS else "reported, not gated"
    print(f"  {_f:16s} -> {_s:20s} ({_tag}; must beat {', '.join(RIVALS[_f])})")
print("censure and moral_balance: no admissible anchor, so clause C cannot be run")
print("on them at all. Consistent with prereg 5.15 (design section 7).")


In [ ]:

# === DO (4 of 15): the bootstrap, which every interval below comes from ===
# WHY RESAMPLING RATHER THAN A FORMULA. Almost every quantity here is computed on a
# small, badly imbalanced slice. The analytic interval for a correlation assumes a
# bivariate normal shape that 0-to-1 facet scores do not have, and there is no
# closed form at all for the interval around a difference between two partial
# correlations, which is what clause C compares. Resampling the observed rows makes
# no distributional claim: it asks what this statistic would have looked like had we
# drawn a different sample of the same people, and reads the answer off the spread.
RNG = np.random.default_rng(RANDOM_SEED)

# 5000 resamples puts 125 draws in each 2.5% tail, which holds a percentile bound
# steady to about the third decimal. That resolution is the point: clause B compares
# a lower bound against .60 and clause C asks whether two intervals overlap, so tail
# noise would otherwise decide clauses.
N_BOOT = 5000 if RUN_MODE == "full" else 400

# HAZARD: RNG is a single generator shared by every bootstrap below. The numbers are
# reproducible only when the cells are run in order, top to bottom, from a fresh
# runtime. Re-running one clause cell on its own advances the stream and shifts the
# bounds in the third decimal, which is enough to flip a borderline facet.

def boot_ci(stat_fn, *arrays, n_boot=None, alpha=0.05):
    """Paired bootstrap interval for any statistic. Returns (estimate, lo, hi).

    Rows are resampled JOINTLY -- one index vector is drawn and applied to every
    array -- so a response keeps its ensemble score, its survey answer and its word
    count together. Drawing each array independently would destroy exactly the
    association being estimated.

    The point estimate is computed on the observed rows, never as the mean of the
    resamples, so the reported number is the sample statistic and the interval is
    reported around it.

    A degenerate resample (every drawn row identical, so there is no variance and
    the correlation is undefined) is recorded as NaN and dropped. On the sparse
    facets a handful of those is expected, and an interval computed from the
    survivors is more honest than no interval at all.
    """
    n_boot = N_BOOT if n_boot is None else n_boot
    arrays = [np.asarray(a, dtype=float) for a in arrays]
    n = len(arrays[0])
    assert all(len(a) == n for a in arrays), "boot_ci: array length mismatch"
    point = stat_fn(*arrays)
    if n < 3:
        return point, np.nan, np.nan
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = RNG.integers(0, n, n)          # ONE index vector: this is the pairing
        try:
            stats[b] = stat_fn(*[a[idx] for a in arrays])
        except Exception:
            stats[b] = np.nan
    lo, hi = np.nanpercentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lo, hi

print(f"Bootstrap ready: {N_BOOT:,} resamples, seed {RANDOM_SEED} (frozen).")


In [ ]:

# === DO (4 of 15, cont.): reading an interval ===
# Three small helpers, and one rule between them: a quantity that could not be
# computed is shown as -- and never as an empty cell, because an empty cell in a
# gate table is read as a failure and the two mean opposite things.
def ci_nonoverlap(ci_a, ci_b):
    """True when two (lo, hi) intervals do not overlap.

    Clause C's separation test. Requiring two 95% intervals to be disjoint is a
    deliberately CONSERVATIVE criterion, stricter than a two-sample test at p < .05,
    and the conservatism is the intent: clause C decides whether a facet earns
    primary standing, so a facet whose own and rival correlations differ only in
    their point estimates should not clear it.

    A missing bound returns False. An interval that could not be estimated has not
    shown separation, and silence must not pass for evidence.
    """
    (a_lo, a_hi), (b_lo, b_hi) = ci_a, ci_b
    if any(np.isnan(v) for v in (a_lo, a_hi, b_lo, b_hi)):
        return False
    return bool((a_hi < b_lo) or (b_hi < a_lo))

def width(lo, hi):
    """Interval width, printed beside every estimate so a wide pass is visible."""
    return np.nan if (np.isnan(lo) or np.isnan(hi)) else float(hi - lo)

def num(v, nd=3):
    """A number, or two dashes where there is none.

    Every printed table below goes through this. A quantity that could not be
    computed shows as -- rather than as an empty cell, because an empty cell in a
    gate table is read as a failure and the two mean opposite things.
    """
    return "--" if (v is None or (isinstance(v, float) and np.isnan(v))) else f"{v:.{nd}f}"

print("Interval readers ready: disjointness, width, and -- for what has no number.")


In [ ]:

# === DO (5 of 15): partial correlation, written out ===
# WHAT A PARTIAL CORRELATION IS, in one sentence: the correlation between two
# things after the part of each that is predictable from a third thing has been
# subtracted out.
#
# WHY WORD COUNT IS THAT THIRD THING. Prereg 6.7: "word count is a covariate, since
# people differ in how much they write". A longer answer has more chances to trip
# any facet AND more chances to say something a survey scale agrees with, so an
# unpartialled facet-to-scale correlation is partly a measure of verbosity. The item
# medians here run about 32 words for the ranking item, 19 backward and 13 forward,
# so this is not a small effect to leave in.
#
# The computation: regress each variable on the covariates, keep the residuals, and
# correlate those. That is the definition of a partial correlation, and it returns
# the same number as the standard library routines -- it is written out here so the
# notebook needs no extra package and so the bootstrap below can call it tens of
# thousands of times without taking minutes.

def _residuals(y, X):
    """y with the part explained by X (which carries an intercept) removed."""
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ beta

def partial_r(x, y, covars):
    """Partial correlation of x and y given the columns of `covars` (n x k or None)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    n = len(x)
    X = np.ones((n, 1)) if covars is None or np.size(covars) == 0 else \
        np.column_stack([np.ones(n), np.asarray(covars, dtype=float)])
    rx, ry = _residuals(x, X), _residuals(y, X)
    # A zero-variance residual means one of the variables was entirely explained by
    # the covariates, or never varied at all. There is no correlation to report, and
    # returning 0.0 would read as "measured, and null".
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def partial_p(r, n, k):
    """Two-sided p-value for a partial correlation on n rows with k covariates."""
    df = n - 2 - k
    if np.isnan(r) or df <= 0 or abs(r) >= 1.0:
        return np.nan
    tstat = r * np.sqrt(df / (1 - r ** 2))
    return float(2 * t_dist.sf(abs(tstat), df))

def partial_ci(frame, xcol, ycol, covar_cols, min_n=10):
    """Partial correlation with a bootstrap interval. Returns (r, (lo, hi), n).

    Listwise deletion across the pair AND the covariates first, so the estimate, the
    interval and the n all describe the same rows. Fewer than min_n complete rows
    returns nothing rather than a number nobody should read.
    """
    cols = [xcol, ycol] + list(covar_cols)
    sub = frame[cols].apply(pd.to_numeric, errors="coerce").dropna()
    n = len(sub)
    if n < min_n:
        return np.nan, (np.nan, np.nan), n
    xv = sub[xcol].to_numpy(float); yv = sub[ycol].to_numpy(float)
    cv = sub[list(covar_cols)].to_numpy(float) if covar_cols else np.zeros((n, 0))
    def stat(a, b, *cs):
        C = np.column_stack(cs) if cs else None
        return partial_r(a, b, C)
    est, lo, hi = boot_ci(stat, xv, yv, *[cv[:, j] for j in range(cv.shape[1])])
    return est, (lo, hi), n

print("Partial correlation ready (word count is the covariate, prereg 6.7).")


In [ ]:

# === DO (6 of 15): one table, four namespaces, no collisions ===
# WHAT THIS CELL DOES: puts the ensemble scores, the word-list counts and the survey
# scales on one row per response x item, so clause C can correlate one against
# another.
#
# THE NAMESPACE RULE, AND WHY IT IS THE MOST IMPORTANT LINE IN THIS CELL. Six of the
# nine facet names are ALSO survey scale names (revenge, suffering, proportionality,
# deterrence, incapacitation, rehabilitation). A bare facet column and a survey
# column would land on the same name in the merge, and pandas would settle it with
# _x/_y suffixes on whichever frame arrived second. One prefix per source keeps every
# name unambiguous:
#     ens_<facet>     the language-model ensemble mean          (notebook 07)
#     dict_<facet>    the word-list hit count                   (notebook 01)
#     <bare name>     a participant-level survey scale          (retribution_scored)
# Anything unprefixed is survey, WITH FIVE EXCEPTIONS that ride through from
# notebook 01 keeping their own names because nothing on the survey side is called
# any of them: n_words, vignette, orientation, principled_any and vengeful_any.
# The rule is what lets clause C read SURVEY_SCALES as survey scales, and it is
# CHECKED below rather than trusted.
ens_wide = (ens.pivot_table(index=["response_id", "item_type"], columns="facet",
                            values="ensemble_mean").reset_index())
ens_wide.columns = (["response_id", "item_type"]
                    + [f"ens_{c}" for c in ens_wide.columns[2:]])

# The word list takes its prefix before it can meet a survey column. n_words,
# vignette and orientation keep their own names: nothing on the survey side is
# called any of those.
_ren = {}
for f in FACETS:
    if f in dict_df.columns:
        _ren[f] = f"dict_{f}"
    if f"{f}_per100" in dict_df.columns:
        _ren[f"{f}_per100"] = f"dict_{f}_per100"
dict_keep = dict_df.rename(columns=_ren)
# The negation-zeroed copies are dropped here on purpose. Design section 6: negation
# handling is zero-out only, whitelist-based, and EXCLUDED FROM EVERY GATE -- the raw
# counts are what the convergence legs are computed on, so the two versions can never
# be confused for one another in this notebook.
dict_keep = dict_keep[[c for c in dict_keep.columns if not c.endswith("_negzeroed")]]

# THE MERGE, GUARDED ON BOTH SIDES (the contract from LOAD (3 of 3), applied here).
# The right-hand frame must be unique on the join keys or a single ensemble row
# fans out into two, and every count printed below grows without anything saying so.
assert not dict_keep.duplicated(["response_id", "item_type"]).any(), (
    "the word-list frame is not unique on (response_id, item_type); the merge "
    "would fan one ensemble row into several. Re-run notebook 01.")
_n_before = len(ens_wide)
feat = ens_wide.merge(dict_keep, on=["response_id", "item_type"], how="left",
                      suffixes=("", "_dic"))
# The row count is the second layer. It catches what the uniqueness check above
# would catch if that check were ever removed or weakened, and it catches what that
# check cannot: a join type changed from "left" to "outer" or "right", which adds
# rows for word-list keys the ensemble never scored.
assert len(feat) == _n_before, (
    f"the word-list merge changed the row count ({_n_before} -> {len(feat)}). "
    "Every correlation below would then be computed on duplicated rows.")
_clash = [c for c in feat.columns if c.endswith("_dic")]
assert not _clash, f"a word-list column collided with an ensemble column: {_clash}"
print(f"  ensemble + word list: {feat.shape[0]:,} rows x {feat.shape[1]} columns")


In [ ]:

# === DO (6 of 15, cont.): attach the survey, and guard the join ===
# The survey is participant-level: one row per person fans out onto that person's
# ranking, backward and forward rows. That is intended -- the validators are person
# constants -- but it is also why the merge is checked on both sides. The
# one-row-per-participant check happened in LOAD (3 of 3, cont.), where a duplicate
# set survey to None and SURVEY_NOT_RUN to the reason; it is re-asserted here
# because this is the line the duplicate would have damaged.
SURVEY_OK = False
if survey is not None:
    assert not survey.duplicated(["response_id"]).any(), (
        "retribution_scored.csv must hold exactly one row per participant")
    _n_before = len(feat)
    feat = feat.merge(survey, on="response_id", how="left", suffixes=("", "_surv"))
    # Second layer, as on the word-list merge: this fires if the check above is
    # ever weakened, and if the join type is changed from "left" to "outer" or
    # "right", which would add rows for participants who wrote nothing.
    assert len(feat) == _n_before, (
        f"the survey merge changed the row count ({_n_before} -> {len(feat)})")
    print(f"  survey merged onto the item rows: {len(feat):,} rows, "
          f"{feat.shape[1]} columns.")
else:
    print(f"  Clause C cannot run: {SURVEY_NOT_RUN}.")


In [ ]:

# === DO (6 of 15, cont.): the collision guard, and the gate site ===
if survey is not None:
    # THE COLLISION GUARD, and the reason the prefixes above exist. A _surv-suffixed
    # column means a survey column lost its bare name to a text-derived column that
    # was already in the table, and every later reference to that bare name would
    # then read the text column instead of the survey one.
    #
    # WHICH COLLISIONS COST WHAT. Only a collision on a name clause C actually
    # correlates against can corrupt a verdict: prereg 6.7 requires each facet to
    # correlate with its matching SURVEY scale, and `revenge` and `suffering` are
    # both facet names and survey scale names, so on those the clause would go on
    # printing pass and fail while quietly testing the ensemble against a word list.
    # A collision on any OTHER name -- n_words, vignette, orientation,
    # principled_any, vengeful_any -- is a real problem in the export and is printed
    # as one, but it cannot reach a clause C verdict, and taking the clause down for
    # all nine facets over it would be the larger error.
    _collided = [c for c in feat.columns if c.endswith("_surv")]
    _scale_names = set(SURVEY_SCALES) | set(VALIDATOR.values())
    _scale_hits = [c for c in _collided if c[:-len("_surv")] in _scale_names]
    for c in _collided:
        print(f"  COLLISION: the survey's {c[:-len('_surv')]!r} was renamed to {c!r} "
              "because a text-derived column already held that name.")
    if _scale_hits:
        SURVEY_NOT_RUN = (f"survey scale columns collided and were renamed: "
                          f"{_scale_hits}")
        print("  STOP AND READ: those are scales clause C correlates against.")
        print("  Clause C is recorded as NOT RUN rather than computed against the")
        print("  wrong column. Fix the prefixes upstream and re-run.")
    else:
        if _collided:
            print("  None of those is a scale clause C reads, so clause C still")
            print("  runs. Fix the prefixes upstream anyway.")
        SURVEY_OK = True
        _matched = feat["response_id"].isin(set(survey["response_id"])).sum()
        print(f"  survey attached: {_matched:,} of {len(feat):,} item rows carry a "
              "participant match")
        if _matched < 0.5 * len(feat):
            # Reachable because LOAD (3 of 3) put both id columns into text form
            # first. Left as they came off disk, a character id meeting an integer
            # id raises inside the merge two lines above and this never prints.
            print("  WARNING: fewer than half the rows matched. The two files are")
            print("  probably keyed on different id spaces; clause C would then be")
            print("  computed on a small and non-random subset.")

# The gate site: one row per participant at the backward item (design section 3).
gate_df = feat[feat["item_type"] == GATE_ITEM].copy()
# Every clause C correlation treats these rows as independent people. Two rows for
# one participant would count that person twice in every r, every interval and
# every n on the clause C table.
assert not gate_df.duplicated(["response_id"]).any(), (
    f"the {GATE_ITEM} slice holds more than one row for some participant, so "
    "clause C would count those people twice. Re-run notebooks 00, 01 and 07.")
print(f"  gate site ({GATE_ITEM}): {len(gate_df):,} rows, one per participant")


In [ ]:

# === DO (7 of 15): clause A, which is not the registration's clause ===
# WHAT THIS CELL DOES: reads the human-match numbers if the optional human-coding
# notebook produced them, and otherwise records clause A as NOT RUN for every facet.
#
# STANDING OF THIS CLAUSE -- READ THIS BEFORE CITING IT. Clause A is NOT something
# prereg 6.7 requires. The registration's gate is two clauses: "a facet supports a
# claim only if the models agree at an intraclass correlation of at least .60 and
# the facet correlates at least .30 with its matching survey scale". Clause A, the
# human-coded gold standard behind it, the by-stratum reporting and the shared-error
# cap on clause B are ADDITIONS to this design, carried on the dated OSF addendum
# (design section 0). They ADD to the gate and never relax it: a facet that would
# have passed the registered two clauses can only be demoted here, never promoted.
#
# The justification is prereg 6.7's own sentence -- "agreement among the models is
# reliability, not validity" -- which names the gap without closing it. Clause A is
# what closes it, and with HUMAN_GOLD False there is no independent check on shared
# bias anywhere in this chain. That is a limitation of the registered design, and it
# is reported as one rather than hidden by a blank column.
#
# CONTRACT for human_match_results.csv, exactly as notebook 99 writes it. One row
# per (scope, facet), and the scope column says what the row IS:
#   pooled         that facet's clause A verdict, on the whole gold TEST half
#   stratum        the same estimate inside one stratum, reported and not gated
#   stratum_class  the clean-versus-boundary collapse, reported and not gated
# ONLY THE POOLED ROWS CARRY A VERDICT, and filtering on scope is not optional.
# Read without the filter, the last row for a facet wins, the last row is a
# per-stratum row carrying 'reported, not gated', and a facet notebook 99 marked
# FAIL would arrive here with no verdict at all and be reported gate-backed.
A_REQUIRED = ["scope", "facet", "clause_a", "est", "ci_lo", "n_pos"]
# Notebook 99's spelling on the left, this notebook's on the right. Any other value
# is recorded as 'not run' and named in the printout: a verdict word nobody
# recognises must not be silently believed, and must not silently pass either.
A_VERDICT = {"pass": "pass", "fail": "fail", "not run": "not run"}

def _fnum(v):
    """A float, or NaN. Never raises on a blank or a word."""
    return float(pd.to_numeric(v, errors="coerce"))

gateA = {f: {"clauseA": "not run", "clauseA_note": "", "n_test": np.nan,
             "human_match_est": np.nan, "human_match_lo": np.nan,
             "human_match_hi": np.nan, "n_pos_test": np.nan,
             "human_match_clean": np.nan, "human_match_boundary": np.nan,
             "shared_error_flag": "not run", "signed_bias": np.nan,
             "cross_model_err_corr": np.nan} for f in FACETS}

def a_not_run(reason):
    """Write one reason onto every facet's clause A note.

    The note is the only place the exported file says WHY clause A did not run,
    and 'not run' with no reason beside it is the blank this notebook exists to
    avoid. Called from each branch below with that branch's own reason, so the
    exported note is never a stale default describing a different run.
    """
    for _v in gateA.values():
        _v["clauseA_note"] = reason

a_not_run("HUMAN_GOLD is False: the human arm never ran"
          if not HUMAN_GOLD else "HUMAN_GOLD is True; see the run log for why")
print(f"  clause A default: 'not run' on all {len(FACETS)} facets.")


In [ ]:

# === DO (7 of 15, cont.): read the clause A file, pooled rows only ===
A_POOLED = None
if clauseA_in is not None:
    # Reported, never raised. A defect in the OPTIONAL arm's file must not take the
    # registered two-clause gate down with it.
    _missing = [c for c in A_REQUIRED if c not in clauseA_in.columns]
    if _missing:
        a_not_run(f"human_match_results.csv is missing columns {_missing}")
        print(f"  human_match_results.csv is missing columns {_missing}; clause A "
              "stays NOT RUN. Expected: " + ", ".join(A_REQUIRED))
    else:
        A_POOLED = clauseA_in[clauseA_in["scope"].astype(str).str.strip()
                              == "pooled"].copy()
        if not len(A_POOLED):
            a_not_run("human_match_results.csv carries no scope == 'pooled' rows")
            print("  human_match_results.csv carries no scope == 'pooled' rows, so "
                  "it carries no clause A verdict. Clause A stays NOT RUN.")
            A_POOLED = None
        elif A_POOLED.duplicated(["facet"]).any():
            a_not_run("human_match_results.csv holds more than one pooled row per "
                      "facet")
            print("  human_match_results.csv holds more than one pooled row for some "
                  "facet. Clause A stays NOT RUN rather than be decided by whichever")
            print("  row happened to sort last. Re-run notebook 99.")
            A_POOLED = None
elif HUMAN_GOLD:
    a_not_run("HUMAN_GOLD is True but human_match_results.csv is not in the folder")
    print("  HUMAN_GOLD is True but human_match_results.csv is not in the folder.")
    print("  Clause A stays NOT RUN; run notebook 99 to fill it in.")

if A_POOLED is not None:
    _unknown = []
    for _, r in A_POOLED.iterrows():
        f = str(r["facet"])
        if f not in gateA:
            continue
        raw = str(r["clause_a"]).strip().lower()
        v = A_VERDICT.get(raw)
        if v is None:
            _unknown.append(str(r["clause_a"]))
            v = "not run"
        gateA[f].update({
            "clauseA": v, "clauseA_note": str(r.get("note", "")),
            "human_match_est": _fnum(r["est"]), "human_match_lo": _fnum(r["ci_lo"]),
            "human_match_hi": _fnum(r.get("ci_hi", np.nan)),
            "n_pos_test": _fnum(r["n_pos"]), "n_test": _fnum(r.get("n_rows", np.nan))})
    print(f"  clause A carried in for {len(A_POOLED)} facets (gold TEST half only, "
          f"{len(clauseA_in)} rows in the file, pooled rows only).")
    if _unknown:
        print(f"  UNRECOGNISED clause A verdicts, recorded as not run: "
              f"{sorted(set(_unknown))}. Expected one of {sorted(A_VERDICT)}.")


In [ ]:

# === DO (7 of 15, cont.): the clean and boundary halves of clause A ===
# WHY THE STRATA MATTER. The gold sample was drawn stratified, and the mixed-motive
# BOUNDARY cases are the ones that decide the desert-versus-revenge question. "He
# deserves it" is unanimous; the passage where a writer reaches for both desert and
# payback in one sentence is the case this study exists to read, and it is a small
# share of the sample. A facet that matches humans on the clean anchor-adjacent
# responses and falls apart on the boundary has matched the easy half of the
# construct, and has not earned a primary claim. The two numbers do not decide
# anything here -- the verdict is the pooled row -- but they are what a reader has
# to see beside it, so they travel onto the gate table.
if A_POOLED is not None and "stratum_class" in clauseA_in.columns:
    _cls = clauseA_in[clauseA_in["scope"].astype(str).str.strip() == "stratum_class"]
    for _, r in _cls.iterrows():
        f, cls = str(r["facet"]), str(r.get("stratum_class", "")).strip().lower()
        if f in gateA and cls in ("clean", "boundary"):
            gateA[f][f"human_match_{cls}"] = _fnum(r["est"])
    if len(_cls):
        print("  clean/boundary human-match carried in for "
              f"{_cls['facet'].nunique()} facets.")
    else:
        print("  NOTE: human_match_results.csv carries no scope == 'stratum_class'")
        print("  rows, so the by-stratum reading of clause A is blank. The clause is")
        print("  still decided on the pooled row, but the strata are part of what the")
        print("  addendum promises: recompute them in notebook 99 before the")
        print("  write-up quotes clean versus boundary.")
elif A_POOLED is not None:
    print("  NOTE: no stratum_class column in human_match_results.csv, so the")
    print("  clean-versus-boundary reading of clause A is blank. See notebook 99.")


In [ ]:

# === DO (7 of 15, cont.): the shared-error flag, and the gate mode ===
# THE SHARED-ERROR DIAGNOSTIC, which is what caps clause B below. Averaging three
# raters cancels INDEPENDENT error and launders SHARED error. Notebook 99 flags a
# facet when the ensemble's signed bias against the human consensus exceeds .10 --
# a tenth of the 0-to-1 facet scale, large next to the differences E3-E7 test -- or
# when the mean pairwise correlation among the three models' errors exceeds .50, at
# which point averaging has stopped cancelling error and started concentrating it.
# Both thresholds live in notebook 99 (BIAS_MAX, XCORR_MAX) and are not restated as
# numbers here, because a rule applied in two places drifts apart.
if A_POOLED is not None and "shared_error_flag" in A_POOLED.columns:
    for _, r in A_POOLED.iterrows():
        f = str(r["facet"])
        if f in gateA:
            # Only the two words notebook 99 writes are kept. Anything else --
            # including the blank it writes when the diagnostic did not run --
            # becomes 'not run', so the column never carries the string 'nan',
            # which is neither a flag nor an absence and reads as both.
            _fl = str(r["shared_error_flag"]).strip().lower()
            gateA[f]["shared_error_flag"] = _fl if _fl in ("biased", "ok") else "not run"
            gateA[f]["signed_bias"] = _fnum(r.get("signed_bias", np.nan))
            gateA[f]["cross_model_err_corr"] = _fnum(
                r.get("cross_model_err_corr", np.nan))
    _flagged = sorted(f for f, v in gateA.items() if v["shared_error_flag"] == "biased")
    print("  shared-error diagnostic carried in; it caps clause B below. "
          f"Flagged: {_flagged or 'none'}.")
elif A_POOLED is not None:
    # Clause A ran and the diagnostic did not travel with it. Said out loud: the
    # cap is then not applied to any facet, and a reader must not take an
    # unflagged column for a clean diagnostic.
    print("  NOTE: human_match_results.csv carries no shared_error_flag column, so")
    print("  the cap on clause B is NOT APPLIED on this run. Every clause B verdict")
    print("  below rests on the interval alone. See notebook 99.")

def _flag_to_bool(v):
    """Is this facet shared-error flagged? Parsed by name, never by truthiness.

    Notebook 99 prints the flag as the word 'biased' or the word 'ok'. Python
    truthiness would read the string 'ok' as True and cap EVERY facet -- a result
    that looks exactly like a real shared-error finding and would be believed.
    Anything that is neither word is not a flag and does not cap.
    """
    return str(v).strip().lower() in ("biased", "true", "1", "yes")

HUMAN_GATE_ACTIVE = any(v["clauseA"] in ("pass", "fail") for v in gateA.values())
GATE_MODE = "registered+human (A+B+C)" if HUMAN_GATE_ACTIVE else "registered (B+C)"
print(f"Gate mode for this run: {GATE_MODE}")
if not HUMAN_GATE_ACTIVE:
    print("  Clause A is recorded as 'not run' for every facet -- never as a blank,")
    print("  and never as a failure. This is the two-clause gate of prereg 6.7, and")
    print("  with it the shared-error cap on clause B has nothing to read, so no")
    print("  facet can be capped and none is.")


In [ ]:

# === DO (8 of 15): the intraclass correlation, written out ===
# WHAT AN ICC IS, in one sentence: the share of the variation in the scores that is
# real differences between responses rather than differences between the raters
# reading them.
#
# Two-way random effects, absolute agreement, models as raters and responses as
# targets. Two forms, because they answer different questions (prereg 6.7):
#   ICC(2,k)  reliability of the k-model MEAN, which is the score actually used
#             downstream -- the honest reliability of the measure of record. This is
#             the one clause B is stated on.
#   ICC(2,1)  reliability of a SINGLE model, which does not improve just because
#             more raters were bought. Always the smaller of the two, and reported
#             beside ICC(2,k) so a panel of mediocre raters cannot look like a good
#             instrument.
# Absolute agreement rather than consistency: a model that rank-orders responses
# perfectly but sits half a point high is not interchangeable with the others, and
# the mean across models would inherit that offset.
def icc_2way(Y):
    """(ICC(2,k), ICC(2,1)) for a responses x raters matrix. NaN where undefined."""
    Y = np.asarray(Y, dtype=float)
    n, k = Y.shape
    if n < 5 or k < 2:
        return np.nan, np.nan
    grand = Y.mean()
    row_dev = (Y.mean(axis=1) - grand) ** 2
    col_dev = (Y.mean(axis=0) - grand) ** 2
    # Mean squares: between responses, between raters, and what is left over.
    msr = k * row_dev.sum() / (n - 1)
    msc = n * col_dev.sum() / (k - 1)
    sse = ((Y - grand) ** 2).sum() - k * row_dev.sum() - n * col_dev.sum()
    mse = sse / ((n - 1) * (k - 1))
    den_k = msr + (msc - mse) / n
    den_1 = msr + (k - 1) * mse + k * (msc - mse) / n
    icc2k = (msr - mse) / den_k if den_k > 0 else np.nan
    icc21 = (msr - mse) / den_1 if den_1 > 0 else np.nan
    return float(icc2k), float(icc21)

def model_matrix(facet):
    """Rows the whole panel scored, one column per model.

    LISTWISE across models, deliberately. An ICC is a statement about a fixed panel
    of raters, so it has to be estimated on responses the whole panel actually
    rated. Where one model's coverage was poor this frame gets small, and that shows
    up as a wide interval rather than as a silently different quantity.
    """
    cols = [f"score_{m}" for m in MODELS]
    sub = ens[ens["facet"] == facet][cols].apply(pd.to_numeric, errors="coerce")
    return sub.dropna().to_numpy(float)

# WHY AN ICC ALONE IS NOT ENOUGH, which is what the next cell's cap is for. An ICC
# cannot tell agreement from correlated error. Three models drawn from overlapping
# training corpora can be reliably wrong in the same direction, and the direction at
# issue is exactly the desert-versus-revenge line this study tests, where a shared
# prior about how punishment talk works would do the most damage. So clause B below
# is conjunctive: the interval must clear .60 AND the facet must not carry notebook
# 99's shared-error flag. The cap can only demote a facet, never promote one, and
# with HUMAN_GOLD False there is no flag to read and nothing is capped (design
# section 4, rule 7 of section 9).
print("ICC defined. It is checked against notebook 07's value in the next cell.")


In [ ]:

# === DO (9 of 15): clause B, the registered reliability clause ===
# THE REGISTERED HALF IS THE PREREG'S OWN, unchanged: "the models agree at an
# intraclass correlation of at least .60". The .60 floor is not an implementer's
# choice. What this design adds is the interval -- the clause is decided on the
# lower bound, with the point estimate the prereg names reported beside it -- and
# the SHARED-ERROR CAP, whose reasoning is in the cell above.
#
# TWO ROUTES TO ONE NUMBER. The point estimate is ALSO carried in from notebook 07,
# which computed it with a different library. The two are compared below. If they
# disagree by more than a thousandth, one of the two routes is wrong and the whole
# clause is in doubt -- exactly the kind of error a single implementation hides.
# The carried value is kept in its own column, never mixed into the recomputed one.
icc_boot = {}
_drift = []
# One presence rule, not two: the clear-presence threshold notebook 07 wrote must
# be the one this notebook uses.
if "present_clear_threshold" in icc_in.columns and len(icc_in):
    assert np.isclose(float(icc_in["present_clear_threshold"].iloc[0]), PRESENT_CLEAR), (
        "notebooks 07 and 10 hold different PRESENT_CLEAR values; the presence "
        "definition is one rule, fixed in design section 0.")
for f in FACETS:
    row = icc_in[icc_in["facet"] == f]
    carried = float(row["icc2k"].iloc[0]) if len(row) else np.nan
    note_07 = row["clauseB_note"].iloc[0] if len(row) and "clauseB_note" in row else ""
    status_07 = row["clauseB"].iloc[0] if len(row) and "clauseB" in row else "not run"
    biased = _flag_to_bool(gateA[f]["shared_error_flag"])
    # THE POSITIVE-CASE FLOOR, carried from notebook 07 so it is counted once.
    # n_positive is the number of responses with any model's score above zero
    # (trace presence); unknown counts leave the floor unapplied rather than
    # inventing a number.
    npos_07 = (float(row["n_positive"].iloc[0])
               if len(row) and "n_positive" in row and pd.notna(row["n_positive"].iloc[0])
               else np.inf)
    floor_07 = (float(row["min_positive_cases"].iloc[0])
                if len(row) and "min_positive_cases" in row
                and pd.notna(row["min_positive_cases"].iloc[0]) else 30.0)
    Y = model_matrix(f)
    n_used, k = (Y.shape if Y.size else (0, 0))
    if n_used < 5 or k < 2:
        # Not run, in words, with the reason. A facet nobody could measure is not a
        # facet that failed, and the difference matters more here than anywhere.
        # icc2k is NaN, NOT notebook 07's carried value: a point estimate printed
        # beside a "not run" verdict reads as a quantity this notebook computed and
        # then declined to judge. The carried value keeps its own column below.
        icc_boot[f] = {"clauseB": "not run", "icc2k": np.nan, "icc21": np.nan,
                       "lo": np.nan, "hi": np.nan, "n_complete": n_used,
                       "note": (str(note_07) if str(note_07) not in ("", "nan")
                                else f"only {n_used} responses scored by every model"),
                       "icc2k_from_07": carried, "clauseB_07_pointwise": status_07,
                       "capped": False}
        continue
    est, est21 = icc_2way(Y)
    _, lo, hi = boot_ci(lambda idx: icc_2way(Y[idx.astype(int)])[0],
                        np.arange(n_used))
    if not np.isnan(carried) and not np.isnan(est) and abs(carried - est) > 0.001:
        _drift.append((f, carried, est))
    if np.isnan(est):
        status, note = "not run", "ICC undefined (no variance between responses)"
    elif np.isnan(lo):
        status, note = "not run", "interval undefined on this sample"
    elif npos_07 < floor_07:
        # APPLIED TO THE VERDICT, NOT ONLY PRINTED. An ICC over responses every
        # model scored zero comes back near 1.000 and would PASS below on the
        # strength of the zeros. Notebook 07 already calls that "not run"; without
        # this branch the gate would overrule it (design section 0, rows 8 and 12).
        status, note = "not run", (f"only {int(npos_07)} responses carry any model "
                                   f"score above zero, below the {int(floor_07)}-case "
                                   "floor -- an agreement about zeros")
    elif lo >= ICC_FLOOR and not biased:
        status, note = "pass", ""
    else:
        bits = []
        if lo < ICC_FLOOR:
            bits.append(f"interval reaches below the registered {ICC_FLOOR:.2f}"
                        + ("; the point estimate alone would have passed"
                           if est >= ICC_FLOOR else ""))
        if biased:
            # The interesting case: the models agreed, and the diagnostic says that
            # agreement is partly a shared error, so the facet is held to
            # descriptive standing despite clearing the registered threshold.
            bits.append("CAPPED by the shared-error flag: signed bias or cross-model "
                        "error correlation past notebook 99's thresholds")
        status, note = "fail", "; ".join(bits)
    icc_boot[f] = {"clauseB": status, "icc2k": est, "icc21": est21, "lo": lo,
                   "hi": hi, "n_complete": n_used, "note": note,
                   "icc2k_from_07": carried, "clauseB_07_pointwise": status_07,
                   "capped": bool(biased)}


In [ ]:

# === DO (9 of 15, cont.): read clause B ===
print("CLAUSE B -- do the models agree with each other?")
print(f"(prereg 6.7: pass at ICC(2,k) >= {ICC_FLOOR:.2f}, AND not shared-error "
      f"flagged. Decided on the lower bound of a {N_BOOT:,}-resample 95% interval. "
      f"Raters: {len(MODELS)})\n")
print(f"{'facet':16s} {'B':>8s} {'ICC(2,k)':>9s} {'95% interval':>18s} {'width':>7s} "
      f"{'ICC(2,1)':>9s} {'n':>6s} {'shared err':>10s}   note")
print("-" * 120)
for f in FACETS:
    b = icc_boot[f]
    ci = (f"[{b['lo']:.3f}, {b['hi']:.3f}]" if not np.isnan(b["lo"]) else "--")
    est, w, one = num(b["icc2k"]), num(width(b["lo"], b["hi"])), num(b["icc21"])
    print(f"{f:16s} {b['clauseB']:>8s} {est:>9s} {ci:>18s} {w:>7s} {one:>9s} "
          f"{b['n_complete']:6d} {gateA[f]['shared_error_flag']:>10s}   {b['note']}")
print("\n  The shared-error column reads 'not run' on every row unless the optional")
print("  human arm supplied it. 'biased' caps the facet: the ICC is then partly")
print("  reading a shared blind spot as consensus, and no ICC repairs that.")

if _drift:
    print("\n  DISAGREEMENT WITH NOTEBOOK 07 on the ICC point estimate:")
    for f, carried, est in _drift:
        print(f"    {f}: notebook 07 says {carried:.4f}, recomputed here {est:.4f}")
    print("  Two independent routes to one quantity have diverged. Do not read the")
    print("  clause B column until that is resolved.")
else:
    print("\n  Point estimates match notebook 07's to within .001 on every facet.")

# WHAT THIS TABLE IS AND WHAT IT IS NOT. Every number above is agreement of the
# models WITH EACH OTHER. That is consistency, not accuracy. Prereg 6.7 draws the
# line itself: "agreement among the models is reliability, not validity". Three
# models trained on overlapping text can read "pay for what he did" as pure desert
# where many writers mean payback, produce three matching numbers and a high ICC,
# and be confidently wrong together. This table cannot tell the difference, and with
# HUMAN_GOLD False nothing else in the chain can either.
print("\n  A wide interval on a high estimate is the pattern to watch for: it means")
print("  the sample is too small to tell a reliable facet from a lucky one.")


In [ ]:

# === DO (10 of 15): clause C, part 1 -- standardize within vignette ===
# WHY. Each participant read one of three randomly assigned vignettes (prereg 3.2),
# and the three differ in circumstance and therefore in how severe and how wordy the
# responses run. Without standardizing within vignette, a facet that simply fires
# more on the harshest vignette would correlate with any survey scale that also runs
# high there -- a between-vignette artifact wearing the costume of construct
# validity. Sentence1_z arrives already standardized within vignette from the R
# pipeline; this is the matching step on the text side.
VIG_COL = next((c for c in ["vignette", "Vignette"] if c in gate_df.columns), None)
CLAUSE_C_FACETS = [f for f in GATE_FACETS + REPORT_ONLY_FACETS if f in VALIDATOR]

if VIG_COL is not None:
    for f in CLAUSE_C_FACETS:
        col = f"ens_{f}"
        if col in gate_df.columns:
            # Population standard deviation within each vignette group. A group with
            # no variation is mapped to zeros rather than to missing, so a facet that
            # never fires on one vignette does not delete that vignette's rows.
            gate_df[col] = gate_df.groupby(VIG_COL)[col].transform(
                lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0
                else s * 0.0)
    print(f"  Facet scores standardized within {VIG_COL} before the correlations.")
else:
    # Not fatal, but it changes what the clause means. Say so here and in the
    # write-up rather than letting the omission pass silently.
    print("  WARNING: no vignette column at the gate site, so facet scores are left")
    print("  RAW. Report that as a limitation: a vignette-level difference in")
    print("  severity can now masquerade as facet-to-survey convergence.")

# Word count is the covariate, per prereg 6.7.
COVAR = ["n_words"] if "n_words" in gate_df.columns else []
if not COVAR:
    print("  WARNING: no n_words column, so word count is NOT partialled out. Prereg")
    print("  6.7 makes it a covariate; without it these correlations partly measure")
    print("  how much each person wrote.")

# WHICH SCALES ARE REALLY SURVEY SCALES. Every name clause C is about to correlate
# against has to have come off retribution_scored.csv. `revenge` and `suffering` are
# both facet names and survey scale names, so this is the one place where a
# text-derived column could stand in for a survey scale and the clause would go on
# printing pass and fail exactly as though nothing were wrong. Three checks per
# name: it exists on the survey file, it survived the merge, and it was not pushed
# aside into a <name>_surv column by a collision. The third is what makes the
# substitution impossible rather than merely unlikely.
SCALES_OK = []
for s in sorted(set(SURVEY_SCALES) | set(VALIDATOR.values())):
    if f"{s}_surv" in gate_df.columns:
        print(f"  scale {s!r} in this table is NOT the survey scale: the survey copy "
              f"was suffixed to {s}_surv by a name collision. Every clause C test "
              "that needs it is recorded as not run.")
    elif s in SURVEY_COLS and s in gate_df.columns:
        SCALES_OK.append(s)
    else:
        print(f"  scale {s!r} is not available as a survey column; every clause C "
              "test that needs it is recorded as not run.")
print("  Verified as participant-level survey columns:", ", ".join(SCALES_OK) or "none")


In [ ]:

# === DO (11 of 15): clause C, part 2 -- every facet against every scale ===
# WHAT THIS CELL DOES: correlates each facet's ensemble score at the gate item
# against every candidate survey scale, word count partialled out, and puts a
# bootstrap interval around each one. It decides nothing -- the verdicts are read
# off these numbers in the next cell.
#
# THE DIAGONAL AND THE OFF-DIAGONAL. The diagonal is the facet against its own
# preregistered scale, which is prereg 6.7's "correlates at least .30 with its
# matching survey scale". The off-diagonal cells are the same facet against the
# other scales, and every one of them is computed -- including the near-orthogonal
# proportionality_jd -- so that the exported matrix carries the preregistered
# "own beats unrelated" comparison as well as the stricter one this design decides
# on. The whole matrix goes out to disk so the off-diagonal is auditable rather
# than summarized away.
#
# RUNTIME: the slowest cell in the notebook. Each cell of the matrix is a partial
# correlation resampled N_BOOT times.
cC_raw = {}
gate_c_rows = []
CLAUSE_C_NOT_RUN = ""      # the reason clause C did not run, or "" if it did
if SURVEY_OK and SCALES_OK:
    for facet in CLAUSE_C_FACETS:
        fcol, own = f"ens_{facet}", VALIDATOR[facet]
        if fcol not in gate_df.columns or own not in SCALES_OK:
            cC_raw[facet] = {"reason": f"no usable column for {facet} or {own}"}
            continue
        r_own, ci_own, n_own = partial_ci(gate_df, fcol, own, COVAR)
        gate_c_rows.append({"facet": facet, "survey_scale": own, "is_own": True,
                            "partial_r": r_own, "ci_lo": ci_own[0],
                            "ci_hi": ci_own[1], "ci_width": width(*ci_own),
                            "n": n_own})
        rivals = {}
        for s in SCALES_OK:
            if s == own:
                continue
            r_s, ci_s, n_s = partial_ci(gate_df, fcol, s, COVAR)
            rivals[s] = {"r": r_s, "ci": ci_s, "n": n_s}
            gate_c_rows.append({"facet": facet, "survey_scale": s, "is_own": False,
                                "partial_r": r_s, "ci_lo": ci_s[0],
                                "ci_hi": ci_s[1], "ci_width": width(*ci_s),
                                "n": n_s})
        cC_raw[facet] = {"own_scale": own, "r_own": r_own, "ci_own": ci_own,
                         "n_own": n_own, "rivals": rivals}
        print(f"  {facet:16s} r with {own:20s} = "
              + (f"{r_own:.3f}" if not np.isnan(r_own) else "  --")
              + f"  (n = {n_own})")
else:
    # The reason has to be the ACTUAL reason. SURVEY_OK goes False on three
    # different faults -- no file, a duplicated participant, a collision on a scale
    # column -- and printing "no survey file in the folder" after a run log that
    # said "[ok] retribution_scored.csv" and "survey scaffold: 948 participants"
    # leaves the reader with two contradictory statements and no way to choose.
    # SURVEY_NOT_RUN carries the reason from wherever the fault was found.
    CLAUSE_C_NOT_RUN = (SURVEY_NOT_RUN if not SURVEY_OK
                        else "no admissible survey scale columns at the gate site")
    print(f"  Clause C not computed: {CLAUSE_C_NOT_RUN}.")


In [ ]:

# === DO (12 of 15): clause C, part 3 -- the rival it must beat ===
# WHAT THIS CELL DOES: for each facet, picks the rival scale it has to out-correlate
# and asks whether the survey is even able to draw that boundary. No verdicts yet.
cC_tests = {}
for facet in CLAUSE_C_FACETS:
    d = cC_raw.get(facet)
    if not d or "r_own" not in d:
        continue
    r_own, ci_own, rivals = d["r_own"], d["ci_own"], d["rivals"]

    # The rival is picked from the REQUIRED (confusable) list, and the STRONGEST of
    # them, so the clause is decided by the hardest available comparison. The -9
    # sentinel keeps a missing rival from winning the maximum and hiding a real one.
    req = [s for s in RIVALS[facet] if s in rivals]
    rival_missing = ""
    if req:
        top = max(req, key=lambda s: (rivals[s]["r"] if not np.isnan(rivals[s]["r"])
                                      else -9))
        r_top = rivals[top]["r"]
        delta = r_own - r_top
        nonoverlap = ci_nonoverlap(ci_own, rivals[top]["ci"])
    else:
        # NO REQUIRED RIVAL IN THE DATA, which means the discriminant half of the
        # clause CANNOT BE EVALUATED. That is "not run", not "fail". The two are
        # opposite statements: a fail says the facet correlates about as well with
        # the scale it is supposed to be distinguished FROM, which is a finding
        # about the facet; a not run says the scale it had to be distinguished from
        # is not in the data, which is a finding about the file. Recorded here and
        # honoured in the verdict cell below.
        top, r_top, delta, nonoverlap = None, np.nan, np.nan, False
        rival_missing = ("the required rival scale(s) "
                         + ", ".join(RIVALS[facet]) + " are not in the survey data")

    # COLLINEARITY. If the facet's own scale and its top rival are themselves
    # near-copies, "beating the rival" says something about the survey and nothing
    # about the facet. r(deserved, revenge) on this instrument runs about .61, right
    # at the floor, so this branch is live rather than hypothetical -- and it is
    # exactly why the revenge unique-component test below exists. The facet is
    # demoted rather than credited: the survey gate cannot certify a distinction the
    # survey itself does not draw.
    collinear = False
    if top is not None:
        sub2 = gate_df[[d["own_scale"], top]].dropna()
        if len(sub2) >= 10 and sub2[d["own_scale"]].std() > 0 and sub2[top].std() > 0:
            if abs(pearsonr(sub2[d["own_scale"]], sub2[top])[0]) >= COLLINEAR_FLOOR:
                collinear = True

    cC_tests[facet] = {"top": top, "r_top": r_top, "delta": delta,
                       "nonoverlap": nonoverlap, "collinear": collinear,
                       "rival_missing": rival_missing,
                       "unique_state": "not required", "unique_note": ""}


In [ ]:

# === DO (12 of 15, cont.): the one extra test revenge has to survive ===
for facet in CLAUSE_C_FACETS:
    t = cC_tests.get(facet)
    if t is None:
        continue

    # REVENGE'S EXTRA TEST, WHICH IS REQUIRED OF REVENGE AND OF NOTHING ELSE.
    # Revenge carries the heaviest interpretive load in this study, so it faces one
    # more question: with the desert scale partialled out alongside word count, does
    # revenge LANGUAGE still track the revenge SCALE? If revenge language is only
    # desert language in a harder register, this drops below the floor and the
    # clause fails. Decided on the point estimate and its p-value, as the design
    # states it, with the interval printed beside it rather than used -- putting the
    # lower-bound rule on this one as well would demote revenge for a reason the
    # design never stated.
    #
    # THE TEST IS NOT WAIVED WHEN WORD COUNT IS MISSING. n_words only enters as a
    # covariate; without it the test still asks the question that matters, and
    # skipping it on that ground would let revenge clear clause C without facing the
    # one test the design requires of it -- a waiver that can only help it pass.
    # Where `deserved` itself is unavailable the test is "not run", and a revenge
    # verdict is then not available either.
    unique_state = "not required" if facet != "revenge" else "not run"
    unique_note = ""
    if facet == "revenge" and "deserved" in SCALES_OK:
        _cov = list(COVAR) + ["deserved"]
        r_u, ci_u, n_u = partial_ci(gate_df, "ens_revenge", "revenge", _cov)
        p_u = partial_p(r_u, n_u, len(_cov))
        unique_state = ("pass" if bool((not np.isnan(r_u)) and r_u >= R_OWN_FLOOR
                                       and (not np.isnan(p_u)) and p_u < 0.05)
                        else "fail")
        unique_note = (f"revenge given deserved{'' if COVAR else ' (word count NOT '
                       'partialled out: no n_words column)'}: r = {num(r_u)} "
                       f"[{num(ci_u[0])}, {num(ci_u[1])}] w = {num(width(*ci_u))}, "
                       f"p = {'--' if np.isnan(p_u) else f'{p_u:.3g}'}, n = {n_u}")
    elif facet == "revenge":
        unique_note = ("revenge's unique-component test could not be run: the "
                       "`deserved` scale is not an admissible survey column here")

    t["unique_state"], t["unique_note"] = unique_state, unique_note
    if facet == "revenge":
        print(f"  revenge unique-component test: {unique_state}"
              + (f" -- {unique_note}" if unique_note else ""))


In [ ]:

# === DO (13 of 15): clause C, part 4 -- the verdicts ===
# ALL of it, or the clause fails: the diagonal's lower bound clears .30, the margin
# over the most confusable rival clears .15, the two intervals are disjoint,
# revenge's unique component survives, and the survey is able to arbitrate.
#
# FAIL AND NOT RUN ARE DIFFERENT ANSWERS AND ARE KEPT DIFFERENT HERE. A fail says
# the clause was applied and the facet did not clear it. A not run says nobody could
# apply it -- the survey file is absent, the rows are too few, the rival scale the
# facet had to beat is not in the data, or revenge's required extra test had no
# `deserved` column to run against. Every one of those routes to "not run" below,
# with the reason on the row. A clause that could not be applied is not a clause the
# facet failed, and the two must never share a word.
def clause_c_verdict(facet, d, t):
    """('pass' | 'fail' | 'not run', the reason in words) for one facet."""
    r_own, lo_own = d["r_own"], d["ci_own"][0]
    if np.isnan(r_own) or np.isnan(lo_own):
        return "not run", f"too few complete rows to estimate (n = {d['n_own']})"
    if t["rival_missing"]:
        return "not run", ("the discriminant half of the clause could not be "
                           "applied: " + t["rival_missing"])
    if facet == "revenge" and t["unique_state"] == "not run":
        return "not run", t["unique_note"]
    if (lo_own >= R_OWN_FLOOR and (not np.isnan(t["delta"]))
            and t["delta"] >= DELTA_R_FLOOR and t["nonoverlap"]
            and t["unique_state"] in ("pass", "not required")
            and (not t["collinear"])):
        return "pass", t["unique_note"]
    # Every reason it failed, not just the first one. A facet can miss on the floor
    # and on the margin at once, and the two mean different things: the first says
    # it does not converge, the second says it does not discriminate.
    bits = []
    if lo_own < R_OWN_FLOOR:
        bits.append(f"interval reaches below {R_OWN_FLOOR:.2f}"
                    + ("; the point estimate alone would have passed"
                       if r_own >= R_OWN_FLOOR else ""))
    if np.isnan(t["delta"]) or t["delta"] < DELTA_R_FLOOR:
        bits.append(f"margin over {t['top']} below {DELTA_R_FLOOR:.2f}")
    if not t["nonoverlap"]:
        bits.append("intervals overlap")
    if t["unique_state"] == "fail":
        bits.append("no unique component once desert is partialled out")
    if t["collinear"]:
        bits.append(f"own scale and {t['top']} are collinear; the survey cannot "
                    "arbitrate this boundary")
    return "fail", "; ".join(bits + ([t["unique_note"]] if t["unique_note"] else []))


In [ ]:

# === DO (13 of 15, cont.): one clause C row per facet ===
# The three diagnostic columns are WORDS, not True/False. Written as booleans they
# would arrive in the exported file as blanks on every facet the clause never ran
# on, and a blank in a boolean column reads in both R and pandas as False -- so
# collinear_demote would read "not demoted" about a check nobody performed, and
# unique_ok would read "passed" about a test nobody applied.
gateC = {}
NOT_RUN_C = {"r_own": np.nan, "ci_own": (np.nan, np.nan), "top_rival": None,
             "r_rival": np.nan, "delta_r": np.nan, "nonoverlap": "not run",
             "collinear": "not run", "n": 0}
for facet in CLAUSE_C_FACETS:
    d, t = cC_raw.get(facet), cC_tests.get(facet)
    if not d or "r_own" not in d or t is None:
        # unique_ok is 'not required' for every facet but revenge, whatever else
        # went wrong: the extra test is asked of revenge alone, so saying "not run"
        # about the other three would imply a test that was owed and skipped.
        gateC[facet] = dict(NOT_RUN_C, clauseC="not run",
                            own_scale=VALIDATOR[facet],
                            unique_ok=("not run" if facet == "revenge"
                                       else "not required"),
                            note=(d or {}).get("reason", CLAUSE_C_NOT_RUN
                                               or "clause C did not run"))
        continue
    verdict, note = clause_c_verdict(facet, d, t)
    _ran = verdict in ("pass", "fail")
    gateC[facet] = {"clauseC": verdict, "note": note, "r_own": d["r_own"],
                    "ci_own": d["ci_own"], "own_scale": d["own_scale"],
                    "top_rival": t["top"], "r_rival": t["r_top"],
                    "delta_r": t["delta"],
                    "nonoverlap": ("yes" if t["nonoverlap"] else "no") if _ran
                                  else "not run",
                    "collinear": ("yes" if t["collinear"] else "no") if _ran
                                 else "not run",
                    "unique_ok": t["unique_state"],
                    "n": d["n_own"]}


In [ ]:

# === DO (13 of 15, cont.): read clause C ===
print("CLAUSE C -- does the facet track the survey scale it should track?")
# The header states what this run actually did. Printing "word count partialled
# out" on a run where DO (10) has just warned that there is no n_words column would
# describe an analysis nobody performed.
print(f"(prereg 6.7: pass at r >= {R_OWN_FLOOR:.2f}, "
      + ("word count partialled out" if COVAR else
         "WORD COUNT NOT PARTIALLED OUT -- no n_words column, see the warning above")
      + f", at the {GATE_ITEM} item. Decided on the interval's lower bound.)\n")
print(f"{'facet':16s} {'C':>8s} {'own scale':>18s} {'r':>7s} {'95% interval':>18s} "
      f"{'width':>7s} {'rival':>16s} {'r':>7s} {'delta':>7s} {'disjoint':>9s}")
print("-" * 122)
for f in CLAUSE_C_FACETS:
    g = gateC[f]
    ci = (f"[{g['ci_own'][0]:.3f}, {g['ci_own'][1]:.3f}]"
          if not np.isnan(g["ci_own"][0]) else "--")
    print(f"{f:16s} {g['clauseC']:>8s} {g['own_scale']:>18s} "
          f"{num(g['r_own']):>7s} {ci:>18s} {num(width(*g['ci_own'])):>7s} "
          f"{str(g['top_rival'] or '--'):>16s} {num(g['r_rival']):>7s} "
          f"{num(g['delta_r']):>7s} {str(g['nonoverlap']):>9s}")
for f in CLAUSE_C_FACETS:
    if gateC[f]["note"]:
        print(f"  {f}: {gateC[f]['note']}")

print("\n  Facets with NO admissible survey anchor -- censure, moral_balance -- are")
print("  absent from this table on purpose. Prereg 5.15 names four facet-to-scale")
print("  mappings and neither of them is there, and the available proxies each")
print("  correlate .48 to .80 with `deserved`, so pressing one into service would")
print("  launder the desert scale under a new name. They CANNOT be gated on clause")
print("  C, which is consistent with the registration, not a failure of the facet.")
print("  proportionality appears here for the record only: its standing was fixed")
print("  in advance on its ~1% base rate (a registered deviation), and these numbers")
print("  are printed so that demotion is auditable rather than assumed.")


In [ ]:
# === DO (13 of 15, cont.): clause C again, on the literal revenge scale ===
# WHAT THIS CELL DOES: reruns every clause C comparison that uses the survey revenge
# scale with revenge_lit (revenge_1 and _2 only) in its place, and lets a facet pass
# clause C only if it passes BOTH ways (design section 0, the revenge-scale row).
# Two facets are touched: REVENGE, whose own scale and unique-component test use the
# revenge scale, and DESERT, whose confusable rivals include it. Suffering and
# proportionality never reach the revenge scale and keep their verdict unchanged.
#
# WHY. revenge_4, one of the registered scale's four items, asks for harsher
# punishment and says nothing about getting even; on the survey it correlates .78
# with punitiveness. A revenge TEXT facet validated only against that scale is
# partly being validated against punitiveness. The registered verdict is the
# preregistered quantity and stays on the row; the literal verdict sits beside it;
# the COMBINED verdict, the stricter of the two, is what routes the facet.
#
# NOTHING IS RECOMPUTED THAT DO (11) ALREADY HOLDS. DO (11) correlated every facet
# against every scale in SCALES_OK, revenge_lit included, so the literal numbers are
# read out of cC_raw. Only revenge's unique-component test needs one new partial
# correlation. The verdict is read by the SAME clause_c_verdict() as DO (13), so the
# two scorings are judged by one rule.
def _touches_revenge_scale(facet):
    return VALIDATOR.get(facet) == "revenge" or "revenge" in RIVALS.get(facet, [])

LIT_SCALE = "revenge_lit"
clause_c_literal = {}
for facet in CLAUSE_C_FACETS:
    if not _touches_revenge_scale(facet):
        clause_c_literal[facet] = {"verdict": "not applicable", "note": ""}
        continue
    d = cC_raw.get(facet)
    if LIT_SCALE not in SCALES_OK:
        clause_c_literal[facet] = {"verdict": "not run",
                                   "note": "revenge_lit is not an admissible survey column"}
        continue
    if not d or "r_own" not in d or (facet == "revenge" and LIT_SCALE not in d["rivals"]):
        clause_c_literal[facet] = {"verdict": "not run",
                                   "note": (d or {}).get("reason", "clause C did not run")}
        continue
    if facet == "revenge":
        _l = d["rivals"][LIT_SCALE]
        d_lit = {"own_scale": LIT_SCALE, "r_own": _l["r"], "ci_own": _l["ci"],
                 "n_own": _l["n"],
                 "rivals": {s: v for s, v in d["rivals"].items() if s != LIT_SCALE}}
    else:
        d_lit = {"own_scale": d["own_scale"], "r_own": d["r_own"],
                 "ci_own": d["ci_own"], "n_own": d["n_own"], "rivals": d["rivals"]}
    # The required rivals, with the registered revenge scale swapped for the literal.
    req = [LIT_SCALE if s == "revenge" else s for s in RIVALS[facet]]
    req = [s for s in req if s in d_lit["rivals"]]
    t_lit = {"unique_state": "not required", "unique_note": "", "rival_missing": ""}
    if req:
        top = max(req, key=lambda s: (d_lit["rivals"][s]["r"]
                                      if not np.isnan(d_lit["rivals"][s]["r"]) else -9))
        t_lit.update(top=top, r_top=d_lit["rivals"][top]["r"],
                     delta=d_lit["r_own"] - d_lit["rivals"][top]["r"],
                     nonoverlap=ci_nonoverlap(d_lit["ci_own"],
                                              d_lit["rivals"][top]["ci"]))
        sub2 = gate_df[[d_lit["own_scale"], top]].dropna()
        t_lit["collinear"] = bool(
            len(sub2) >= 10 and sub2.iloc[:, 0].std() > 0 and sub2.iloc[:, 1].std() > 0
            and abs(pearsonr(sub2.iloc[:, 0], sub2.iloc[:, 1])[0]) >= COLLINEAR_FLOOR)
    else:
        t_lit.update(top=None, r_top=np.nan, delta=np.nan, nonoverlap=False,
                     collinear=False,
                     rival_missing=("the required rival scale(s) "
                                    + ", ".join(RIVALS[facet])
                                    + " are not in the survey data"))
    if facet == "revenge":
        if "deserved" in SCALES_OK:
            _cov = list(COVAR) + ["deserved"]
            r_u, ci_u, n_u = partial_ci(gate_df, "ens_revenge", LIT_SCALE, _cov)
            p_u = partial_p(r_u, n_u, len(_cov))
            t_lit["unique_state"] = ("pass" if (not np.isnan(r_u)) and r_u >= R_OWN_FLOOR
                                     and (not np.isnan(p_u)) and p_u < 0.05 else "fail")
            t_lit["unique_note"] = (f"revenge_lit given deserved: r = {num(r_u)} "
                                    f"[{num(ci_u[0])}, {num(ci_u[1])}], "
                                    f"p = {'--' if np.isnan(p_u) else f'{p_u:.3g}'}, "
                                    f"n = {n_u}")
        else:
            t_lit["unique_state"] = "not run"
            t_lit["unique_note"] = ("the unique-component test on revenge_lit could not "
                                    "run: `deserved` is not an admissible survey column")
    _v, _note = clause_c_verdict(facet, d_lit, t_lit)
    clause_c_literal[facet] = {"verdict": _v, "note": _note, "r_own": d_lit["r_own"],
                               "top": t_lit["top"], "delta": t_lit["delta"]}

print("CLAUSE C ON THE LITERAL REVENGE SCALE, and the combined verdict that routes")
print("each facet. A facet passes only if it passes on both revenge scorings.\n")
for facet in CLAUSE_C_FACETS:
    reg = gateC[facet]["clauseC"]
    L = clause_c_literal[facet]
    if L["verdict"] == "not applicable":
        combined = reg
    elif "fail" in (reg, L["verdict"]):
        combined = "fail"
    elif reg == "pass" and L["verdict"] == "pass":
        combined = "pass"
    else:
        combined = "not run"
    gateC[facet]["clauseC_registered_revenge_scale"] = (
        reg if L["verdict"] != "not applicable" else "not applicable")
    gateC[facet]["clauseC_literal_revenge_scale"] = L["verdict"]
    if combined != reg:
        gateC[facet]["note"] = "; ".join(
            x for x in [gateC[facet]["note"],
                        f"literal revenge scale: {L['verdict']} ({L['note']})"] if x)
    gateC[facet]["clauseC"] = combined
    print(f"  {facet:16s} registered scale: {reg:8s} literal scale: "
          f"{L['verdict']:15s} -> clause C {combined}")
    if L.get("note") and L["verdict"] not in ("not applicable",):
        print(f"      {L['note']}")


In [ ]:
# === DO (13 of 15, cont.): clause C numbers at the second backward-looking site ===
# WHAT THIS CELL DOES: computes, for reporting only, each clause C facet's
# correlation with its own survey scale and with its required rivals at Q333, the
# ranking explanation, with the same within-vignette standardization, the same word
# count partial and the same bootstrap as at the gate item. Nothing here enters a
# verdict: Q1 is the site of record (design section 0, the gate-site row).
#
# WHY. Q1 is the scarcest site for the E4 pair. A facet scored zero on most Q1
# responses has little variance to correlate with, so a clause C failure at Q1
# could be a failure of the site rather than of the measure. The same numbers at
# Q333, fixed in advance and printed beside the verdict, let a reader tell those
# apart without the gate item ever being chosen after the fact.
SECOND_SITE = "ranking"
SECOND_SITE_REPORT = {}
q2_df = feat[feat["item_type"] == SECOND_SITE].copy()
assert not q2_df.duplicated(["response_id"]).any(), (
    f"the {SECOND_SITE} slice holds more than one row for some participant")
if VIG_COL is not None and VIG_COL in q2_df.columns:
    for f in CLAUSE_C_FACETS:
        col = f"ens_{f}"
        if col in q2_df.columns:
            q2_df[col] = q2_df.groupby(VIG_COL)[col].transform(
                lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0
                else s * 0.0)
Q2_COVAR = [c for c in COVAR if c in q2_df.columns]
if SURVEY_OK and SCALES_OK:
    print(f"CLAUSE C NUMBERS AT {SECOND_SITE.upper()} (Q333) -- reporting only; "
          f"{GATE_ITEM} decides")
    for facet in CLAUSE_C_FACETS:
        fcol, own = f"ens_{facet}", VALIDATOR[facet]
        if fcol not in q2_df.columns or own not in SCALES_OK:
            continue
        r_o, ci_o, n_o = partial_ci(q2_df, fcol, own, Q2_COVAR)
        req = [s for s in RIVALS[facet] if s in SCALES_OK]
        rv = {s: partial_ci(q2_df, fcol, s, Q2_COVAR) for s in req}
        top = (max(req, key=lambda s: -9 if np.isnan(rv[s][0]) else rv[s][0])
               if req else None)
        rec = {"r_own": r_o, "lo": ci_o[0], "hi": ci_o[1], "n": n_o, "top_rival": top,
               "r_rival": rv[top][0] if top else np.nan,
               "delta": (r_o - rv[top][0]) if top else np.nan}
        if own == "revenge" and "revenge_lit" in SCALES_OK:
            r_l, ci_l, _ = partial_ci(q2_df, fcol, "revenge_lit", Q2_COVAR)
            rec.update(r_own_literal=r_l, lo_literal=ci_l[0], hi_literal=ci_l[1])
        SECOND_SITE_REPORT[facet] = rec
        print(f"  {facet:16s} r with {own:12s} = {num(r_o)} [{num(ci_o[0])}, "
              f"{num(ci_o[1])}]"
              + (f"   rival {top} = {num(rec['r_rival'])}, delta = {num(rec['delta'])}"
                 if top else "")
              + (f"   (literal revenge scale: {num(rec['r_own_literal'])})"
                 if "r_own_literal" in rec else ""))
else:
    print(f"  Second-site numbers not computed: "
          f"{CLAUSE_C_NOT_RUN or SURVEY_NOT_RUN or 'no admissible survey scales'}.")


In [ ]:

# === DO (14 of 15): the agreement matrix, part 1 -- the legs ===
# WHAT THIS TABLE IS FOR. The gate is decided facet by facet, but the question
# behind it -- is this facet measured, or only asserted -- is answered by how the
# three independent scorers line up on the same responses. One row per facet per
# item, 9 x 3 = 27, with the legs side by side:
#     inter-model    ICC(2,k) and its interval, plus Krippendorff's alpha, a
#                            measure of how far independent raters agree, computed
#                            on the models' one-word holistic call rather than on
#                            the 0-to-1 scores
#     ensemble x word list   the more independent check, because the word list uses
#                            no model at all -- reported only above the 5% coverage
#                            floor, since a correlation on three positive cases is
#                            noise whose silence reads as agreement
#     ensemble x embedding   a convergence check that gates nothing
#     ensemble x human       the headline validity leg, and NOT RUN by default
def cov_lookup(facet, item):
    """Coverage for this facet at this item. Returns (pct, verdict, why).

    The verdict is CARRIED from notebook 02 rather than recomputed. The 5% floor is
    that notebook's job, and a rule applied in two places is a rule whose two copies
    eventually disagree.
    """
    if coverage is None:
        return np.nan, "not run", "word_list_coverage.csv is not in the folder"
    row = coverage[(coverage["facet"] == facet) & (coverage["scope"] == item)]
    if not len(row):
        return np.nan, "not run", "no coverage row for this facet at this item"
    pct = row["pct_nonzero"].iloc[0]
    clears = str(row["clears_floor"].iloc[0]).strip().lower()
    pct = float(pct) if pd.notna(pct) else np.nan
    if clears == "yes":
        return pct, "usable", ""
    if clears == "no":
        return pct, "below floor", ("the word list fires too rarely here to be an "
                                    "independent leg for this facet")
    return pct, "not run", "no non-blank responses in this slice"

def leg_r(frame, xcol, ycol, method="pearson", min_n=10):
    """Correlation between two streams on the rows where both exist."""
    sub = frame[[xcol, ycol]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(sub) < min_n or sub[xcol].std() == 0 or sub[ycol].std() == 0:
        return np.nan, len(sub)
    fn = pearsonr if method == "pearson" else spearmanr
    return float(fn(sub[xcol], sub[ycol])[0]), len(sub)

# ANTI-CIRCULARITY on the embedding leg. The reference passages notebook 04 built
# come from the codebook's written anchors AND from dictionary-positive responses in
# the SEED half. A similarity computed on a SEED row is therefore partly a check of
# a stream against the very responses that seeded it. Every embedding number below
# is computed with the SEED rows removed, and the count is reported beside it.
sim_use, _enc = None, []
if sim is not None:
    if "gold_split" in sim.columns:
        sim_use = sim[sim["gold_split"] != "SEED"].copy()
    else:
        # No split label means the SEED rows cannot be removed, so the leg is
        # reported with that stated rather than quietly computed on seeded rows.
        sim_use = sim.copy()
        print("  WARNING: facet_similarity.csv carries no gold_split column, so the")
        print("  SEED rows cannot be held out. Read the embedding leg as partly a")
        print("  check of the embeddings against the responses that seeded them.")
    _enc = [c for c in sim_use.columns if c.endswith("_sim")]
    print(f"  embedding leg: {len(_enc)} encoder(s), SEED rows excluded "
          f"({len(sim):,} rows in, {len(sim_use):,} used)")
else:
    print("  embedding leg: not run, facet_similarity.csv is not in the folder.")


In [ ]:

# === DO (14 of 15, cont.): part 2 -- the reliability and word-list legs ===
# Krippendorff's alpha, a measure of how much independent raters agree, carried in
# from notebook 07. It is computed on the holistic call -- the single word each
# model gives the passage as a whole (principled, vengeful, both or neither) --
# which is what prereg 6.7 asks for on the judgment, as distinct from the ICC on
# the nine 0-to-1 facet scores.
KRIPP = (icc_in["kripp_alpha_holistic"].iloc[0]
         if "kripp_alpha_holistic" in icc_in.columns and len(icc_in) else np.nan)
rows = []
for facet in FACETS:
    b = icc_boot[facet]
    for it in ITEM_TYPES:
        sub = feat[feat["item_type"] == it]
        rec = {"facet": facet, "facet_class": FACET_CLASS[facet], "item_type": it,
               # The reliability legs are per facet, pooled across items, exactly as
               # notebook 07 computed them. The value repeats down the three item
               # rows; it is not three separate estimates.
               "icc2k": b["icc2k"], "icc2k_lo": b["lo"], "icc2k_hi": b["hi"],
               "icc2k_ci_width": width(b["lo"], b["hi"]), "icc21": b["icc21"],
               "icc_scope": "per facet, pooled over items",
               "kripp_alpha_holistic": KRIPP, "clauseB": b["clauseB"]}
        pct, cov_verdict, cov_why = cov_lookup(facet, it)
        rec["dict_coverage_pct"] = pct
        rec["dict_leg"] = cov_verdict
        rec["dict_leg_note"] = cov_why
        if cov_verdict == "usable" and f"ens_{facet}" in sub and f"dict_{facet}" in sub:
            r, n = leg_r(sub, f"ens_{facet}", f"dict_{facet}")
            rec["ens_dict_r"], rec["ens_dict_n"] = r, n
        else:
            # Below the floor the leg is not reported as a number at all. Its
            # silence would otherwise read as "no independent contradiction" when
            # what it means is "no independent measurement".
            rec["ens_dict_r"], rec["ens_dict_n"] = np.nan, 0
        rows.append(rec)
print(f"  reliability and word-list legs built for {len(rows)} facet x item rows.")


In [ ]:

# === DO (14 of 15, cont.): part 3 -- the embedding and human legs ===
for rec in rows:
    facet, it = rec["facet"], rec["item_type"]
    sub = feat[feat["item_type"] == it]
    # Embedding leg, per encoder, Spearman. Absolute similarity is encoder-specific
    # -- the Dark Side pipeline saw a headline embedding number swing 40 points
    # across encoders while rank order held at about .71 -- so only the ORDERING is
    # comparable, and Spearman is the coefficient that reads it.
    per_enc, vals = [], []
    if sim_use is not None and f"ens_{facet}" in sub.columns:
        _right = sub[["response_id", "item_type", f"ens_{facet}"]]
        # The right-hand frame of this merge must be one row per (response_id,
        # item_type) or one similarity would be paired with two ensemble scores and
        # the leg's n would grow with nothing saying so.
        assert not _right.duplicated(["response_id", "item_type"]).any(), (
            f"the {it} slice of feat is not unique on (response_id, item_type)")
        s_f = sim_use[sim_use["facet"] == facet].merge(
            _right, on=["response_id", "item_type"], how="inner")
        s_f = s_f[s_f["item_type"] == it]
        for c in _enc:
            r, n = leg_r(s_f, f"ens_{facet}", c, method="spearman")
            per_enc.append(f"{c[:-4]}={r:.3f}" if not np.isnan(r) else f"{c[:-4]}=--")
            if not np.isnan(r):
                vals.append(r)
        rec["ens_emb_n"] = len(s_f)
    else:
        rec["ens_emb_n"] = 0
    rec["ens_emb_rho"] = float(np.median(vals)) if vals else np.nan
    rec["ens_emb_by_encoder"] = "|".join(per_enc) if per_enc else "not run"
    # The human leg. NOT RUN by default, and said in words. n_test and n_pos_test
    # are the base the estimate rests on and they belong beside it: on the sparse
    # facets n_pos_test lands near a dozen, which is exactly why clause A is decided
    # on an interval bound rather than on a point estimate.
    a = gateA[facet]
    rec["ens_human_r"] = a["human_match_est"]
    rec["ens_human_lo"] = a["human_match_lo"]
    rec["ens_human_ci_width"] = width(a["human_match_lo"], a["human_match_hi"])
    rec["human_n_test"] = a["n_test"]
    rec["human_n_pos_test"] = a["n_pos_test"]
    rec["human_match_clean"] = a["human_match_clean"]
    rec["human_match_boundary"] = a["human_match_boundary"]
    # The shared-error flag travels with the ICC because the two have to be read
    # together: a high ICC on a flagged facet is partly shared error.
    rec["shared_error_flag"] = a["shared_error_flag"]
    rec["signed_bias"] = a["signed_bias"]
    rec["cross_model_err_corr"] = a["cross_model_err_corr"]
    rec["ens_human"] = "not run" if a["clauseA"] == "not run" else "computed"
    rec["ens_human_note"] = (
        "HUMAN_GOLD is False: no human coding anywhere in this chain, so the"
        " ensemble has never been compared against a human reading"
        if a["clauseA"] == "not run"
        else "carried from notebook 99, gold TEST half only")
agreement_matrix = pd.DataFrame(rows)
# One row per facet per item, and nothing else. A duplicate here would put the same
# facet on the table twice with two different verdicts beside it.
assert not agreement_matrix.duplicated(["facet", "item_type"]).any(), (
    "the agreement matrix is not unique on (facet, item_type)")
print(f"  agreement matrix: {len(agreement_matrix)} rows "
      f"({len(FACETS)} facets x {len(ITEM_TYPES)} items)")


In [ ]:
# === DO (14 of 15, cont.): the embedding leg's standing, and the evidence for it ===
# DECIDED BEFORE SCORING (design section 0, the embedding-leg row): the embedding
# leg is scored and reported -- absolute levels and cross-encoder rank stability in
# notebook 09, as prereg 6.7 asks, and its agreement with the ensemble above -- but
# it counts as convergent evidence for NO facet.
#
# WHY, IN THREE FACTS.
#   (1) Design section 5 admits a similarity to convergence only once it is
#       human-validated. HUMAN_GOLD is False, so no facet can meet that condition.
#   (2) The only non-human yardstick is the word list, and notebook 04 seeded the
#       reference passages from word-list-positive responses. Testing the leg
#       against word-list hits would test it against its own seeds.
#   (3) Against the survey scales at the gate item the leg carries almost nothing.
#       On the check made before scoring, similarities correlated between -.16 and
#       .13 with each gated facet's own scale, and the revenge similarity correlated
#       NEGATIVELY with the revenge scale on all three encoders.
# This cell recomputes (3) on the run in hand and writes it onto the agreement
# matrix, so the reason for the standing travels with the row.
#
# HOW IT IS COMPUTED. Similarity at the gate item, standardized within vignette as
# the ensemble scores are for clause C, Pearson against the facet's own scale and
# against each of its required rivals, per encoder; the medians across encoders go
# on the row and every encoder's pair is kept in words beside them.
agreement_matrix["emb_convergence_leg"] = "no"
agreement_matrix["emb_role"] = ("scored and reported; not a convergence leg for any "
                                "facet (design section 0)")
agreement_matrix["emb_r_own_scale_median"] = np.nan
agreement_matrix["emb_r_top_rival_median"] = np.nan
agreement_matrix["emb_survey_by_encoder"] = "not run"
if sim_use is not None and SURVEY_OK and _enc:
    print(f"EMBEDDING SIMILARITY AGAINST THE SURVEY AT {GATE_ITEM} "
          "(the reason the leg is not counted as convergence)")
    for facet in CLAUSE_C_FACETS:
        own = VALIDATOR[facet]
        rivals = [s for s in RIVALS[facet] if s in SCALES_OK]
        if own not in SCALES_OK:
            continue
        sf = sim_use[(sim_use["facet"] == facet) & (sim_use["item_type"] == GATE_ITEM)]
        keep = ["response_id"] + ([VIG_COL] if VIG_COL else []) + [own] + rivals
        sf = sf.merge(gate_df[keep], on="response_id", how="inner")
        owns, tops, parts = [], [], []
        for c in _enc:
            if VIG_COL:
                sf[c] = sf.groupby(VIG_COL)[c].transform(
                    lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0
                    else s * 0.0)
            r_o, _n = leg_r(sf, c, own)
            r_rs = [v for v in (leg_r(sf, c, s)[0] for s in rivals) if not np.isnan(v)]
            r_t = max(r_rs) if r_rs else np.nan
            if not np.isnan(r_o):
                owns.append(r_o)
            if not np.isnan(r_t):
                tops.append(r_t)
            parts.append(f"{c[:-4]}: own={num(r_o)} rival={num(r_t)}")
        mask = ((agreement_matrix["facet"] == facet)
                & (agreement_matrix["item_type"] == GATE_ITEM))
        agreement_matrix.loc[mask, "emb_r_own_scale_median"] = (
            float(np.median(owns)) if owns else np.nan)
        agreement_matrix.loc[mask, "emb_r_top_rival_median"] = (
            float(np.median(tops)) if tops else np.nan)
        agreement_matrix.loc[mask, "emb_survey_by_encoder"] = " | ".join(parts)
        print(f"  {facet:16s} against {own:12s} " + " | ".join(parts))
else:
    print("  Embedding-versus-survey check not run: no similarity file, no survey, or")
    print("  no encoder columns. The leg's standing does not depend on it.")


In [ ]:

# === DO (14 of 15, cont.): read the agreement matrix ===
print("AGREEMENT BETWEEN THE THREE SCORERS (facet x item)\n")
_show = ["facet", "item_type", "icc2k", "icc2k_ci_width", "shared_error_flag",
         "dict_coverage_pct", "dict_leg", "ens_dict_r", "ens_emb_rho", "ens_emb_n",
         "emb_convergence_leg", "emb_r_own_scale_median",
         "ens_human", "human_n_pos_test"]
# na_rep is what actually prints a missing number as --. float_format is never
# handed a null by pandas: nulls are rendered through na_rep first, so a fallback
# branch inside float_format is dead code and the table prints the word NaN. This
# notebook's stated rule is that an uncomputable quantity shows as -- and never as
# a blank or a NaN, and na_rep is the argument that makes that rule true.
print(agreement_matrix[_show].to_string(
    index=False, float_format=lambda x: f"{x:.3f}", na_rep="--"))

# HOW TO READ IT, row by row:
print("\nHOW TO READ IT")
print("  high ICC, ens_human 'not run' -> the models agree and nothing has shown")
print("    they are right. Reliability without demonstrated validity, which is")
print("    exactly the gap prereg 6.7 names and the registered design leaves open.")
print("  '--' in ens_dict_r with low coverage -> the word list did not measure this")
print("    facet on this corpus. That is silence, not corroboration.")
print("  a wide icc2k_ci_width -> too few responses behind the estimate to tell a")
print("    reliable facet from a lucky one.")
print("  shared_error_flag 'biased' -> the ICC on that row is partly shared error,")
print("    and clause B caps the facet however high the ICC is.")
print("  human_n_pos_test in single digits -> the human leg's interval will be wide;")
print("    read the bound, not the r.")
print("  ens_emb_rho is a rank correlation and gates nothing. No clause reads it.")
print("  emb_convergence_leg 'no' on every row -> the embedding leg is reported, not")
print("    counted as corroboration for any facet (design section 0). The median")
print("    correlation of the similarity with the facet's own survey scale at the")
print("    gate item is printed beside it as the reason.")


In [ ]:

# === DO (15 of 15): one status per facet ===
# WHAT THE STATUS MEANS, AND WHAT IT DOES NOT
#   GATE_BACKED       cleared every clause that ran, and both registered clauses
#                     ran. Carries primary weight INSIDE the exploratory family.
#   DESCRIPTIVE_ONLY  reported as prevalence, reliability and description. No
#                     primary E3-E7 claim routes through it. Tests it produces
#                     still enter the single Benjamini-Hochberg false-discovery-rate
#                     family -- the multiple-comparison correction that holds the
#                     expected share of false positives among the findings called
#                     significant to 5%, applied once across all of E3 to E7.
#   CONVERGENT_SANITY the three consequentialist goals, scored so retributive
#                     language can be compared against them (prereg 5.15), never
#                     the basis of a retributive claim.
# GATE-BACKED IS NOT CONFIRMATORY. Prereg 2.4: "E3 through E7 are exploratory and
# depend on the validation step in Section 6.7." The gate sets a facet's STANDING
# within that exploratory family; nothing it can return makes a language test
# confirmatory. Every row therefore carries prereg_family = exploratory_E3toE7, so
# no number can be quoted apart from its standing.
def gate_verdict(cA, cB, cC):
    """(passed, reason) for a gate-eligible facet from its three clause words.

    WHICH CLAUSES THIS RUN CAN APPLY. B and C are the preregistered gate and are
    always attempted. A exists only when the optional gold set was coded. A clause
    that never ran is 'not run', NEVER False -- an optional check that was not
    available cannot be allowed to masquerade as a failed one and empty the table.
    """
    ran = [c for c in (cA, cB, cC) if c in ("pass", "fail")]
    if not (cB in ("pass", "fail") and cC in ("pass", "fail")):
        # A gate that could not be applied does not back a claim either. The facet
        # is descriptive-only, and the reason says which clause was missing rather
        # than implying the facet fell short of anything.
        return False, ("the registered gate could not be applied: "
                       + ("clause B not run; " if cB == "not run" else "")
                       + ("clause C not run" if cC == "not run" else "")).strip("; ")
    if all(c == "pass" for c in ran):
        return True, ""
    failed = [n for n, c in zip("ABC", (cA, cB, cC)) if c == "fail"]
    return False, "failed clause " + " and ".join(failed)


In [ ]:

# === DO (15 of 15, cont.): route every facet to its standing ===
standing = {}
for facet in FACETS:
    cA = gateA[facet]["clauseA"]
    cB = icc_boot[facet]["clauseB"]
    cC = gateC.get(facet, {}).get("clauseC", "not run")
    if facet in GATE_FACETS:
        passed, reason = gate_verdict(cA, cB, cC)
        if icc_boot[facet]["capped"]:
            # The reason column is what a reader sees first, and "failed clause B"
            # alone would send them looking for a low ICC that is not there. A
            # capped facet failed clause B with its ICC intact, which is a
            # different finding and has to be legible without opening the notes.
            reason += ("; clause B was CAPPED by the shared-error flag, not missed "
                       "on the ICC")
        status, eligible = ("GATE_BACKED" if passed else "DESCRIPTIVE_ONLY"), "yes"
    elif facet in FORCED_DESCRIPTIVE_ONLY:
        status, eligible = "DESCRIPTIVE_ONLY", "no"
        # The reason string is what distinguishes the two routings in the exported
        # file -- one is consistent with the registration and one is a deviation on
        # the addendum. Keep that distinction intact if this is ever regenerated.
        reason = ("no admissible survey anchor; consistent with prereg 5.15"
                  if facet in ("censure", "moral_balance")
                  else "~1% base rate; descriptive-only -- registered deviation "
                       "from prereg 5.15; the gate numbers are reported anyway")
        if facet in ("censure", "moral_balance"):
            cC = "not run"
    elif facet in CONSEQUENTIALIST:
        # deterrence, incapacitation, rehabilitation. They DO enter E6, where the
        # register shift is the whole point, but no retributive claim routes through
        # them and no gate clause is applied to them. The list is CONSULTED here
        # rather than reached by falling through, so a tenth facet belonging to
        # neither retributive family cannot arrive wearing this standing and this
        # reason line by default. The assert in DO (2) is the other half of that.
        status, eligible = "CONVERGENT_SANITY", "no"
        reason = "consequentialist goal; not a gate target"
        cC = "not run"
    else:
        # Unreachable while the assert in DO (2) holds, and kept because a routing
        # gap must show as a routing gap. A facet with no rule gets the most
        # cautious standing available and a reason that says what happened.
        status, eligible = "DESCRIPTIVE_ONLY", "no"
        reason = ("not routed by any list in DO (2); no standing rule exists for "
                  "this facet, so it is treated as descriptive-only")
        cC = "not run"
    standing[facet] = {"status": status, "gate_eligible": eligible,
                       "reason": reason, "clauseA": cA, "clauseB": cB, "clauseC": cC}


In [ ]:

# === DO (15 of 15, cont.): every number behind every standing, on one row ===
# One row per facet, carrying the verdicts AND the estimates, the intervals and the
# widths they were decided on. A gate table that reports only pass and fail cannot
# be audited: the point estimate the prereg names, the bound this design decides on,
# and the distance between them all have to be on the row.
#
# NO BLANKS IN A VERDICT COLUMN. ci_nonoverlap, collinear_demote and unique_ok are
# words -- yes, no, pass, fail, not run, not required -- and never True/False or
# empty. Written as booleans they arrive empty on the facets clause C never ran on,
# and an empty cell in a boolean column reads as False in both R and pandas: an
# exported claim that a demotion check found nothing, or that a test was passed,
# when in fact nobody ran either.
gate_rows = []
for facet in FACETS:
    A, B = gateA[facet], icc_boot[facet]
    C, S = gateC.get(facet, {}), standing[facet]
    cA, cB, cC = S["clauseA"], S["clauseB"], S["clauseC"]
    gate_rows.append({
        "facet": facet, "facet_class": FACET_CLASS[facet], "status": S["status"],
        "gate_eligible": S["gate_eligible"], "gate_mode": GATE_MODE,
        "gate_item": GATE_ITEM, "prereg_family": "exploratory_E3toE7",
        "reason": S["reason"],
        "clauseA": cA, "clauseA_note": A["clauseA_note"],
        "human_match_est": A["human_match_est"], "human_match_lo": A["human_match_lo"],
        "human_match_ci_width": width(A["human_match_lo"], A["human_match_hi"]),
        "n_pos_test": A["n_pos_test"], "n_test": A["n_test"],
        # The by-stratum reading. A facet that matches the coders on the clean
        # anchor-adjacent responses and falls apart on the contested boundary has
        # matched the easy half of the construct and has not earned a primary claim.
        "human_match_clean": A["human_match_clean"],
        "human_match_boundary": A["human_match_boundary"],
        # The shared-error diagnostic, on the row with the clause it caps.
        "shared_error_flag": A["shared_error_flag"],
        "signed_bias": A["signed_bias"],
        "cross_model_err_corr": A["cross_model_err_corr"],
        "clauseB": cB, "icc2k": B["icc2k"], "icc2k_lo": B["lo"], "icc2k_hi": B["hi"],
        "icc2k_ci_width": width(B["lo"], B["hi"]), "icc21": B["icc21"],
        # Notebook 07's own point estimate and its own reading of it, kept in their
        # own columns. The icc2k column above is what THIS notebook computed, and
        # the two are never mixed into one number with nothing saying which is which.
        "icc2k_from_07": B["icc2k_from_07"],
        "clauseB_07_pointwise": B["clauseB_07_pointwise"],
        "clauseB_capped_by_shared_error": "yes" if B["capped"] else "no",
        "n_complete_icc": B["n_complete"], "clauseB_note": B["note"],
        "icc_floor": ICC_FLOOR, "r_floor": R_OWN_FLOOR,
        "clauseC": cC, "own_scale": C.get("own_scale", ""),
        "r_own": C.get("r_own", np.nan),
        "r_own_lo": C.get("ci_own", (np.nan, np.nan))[0],
        "r_own_hi": C.get("ci_own", (np.nan, np.nan))[1],
        "r_own_ci_width": width(*C.get("ci_own", (np.nan, np.nan))),
        "top_rival": C.get("top_rival") or "not run",
        "r_rival": C.get("r_rival", np.nan),
        "delta_r": C.get("delta_r", np.nan),
        "ci_nonoverlap": C.get("nonoverlap", "not run"),
        "collinear_demote": C.get("collinear", "not run"),
        "unique_ok": C.get("unique_ok", "not required"), "n_gate": C.get("n", 0),
        "clauseC_note": C.get("note", ""),
        # Clause C on each revenge scoring (design section 0). clauseC above is the
        # combined verdict that routed the facet; these two are what it combined.
        "clauseC_registered_revenge_scale": C.get("clauseC_registered_revenge_scale",
                                                  "not applicable"),
        "clauseC_literal_revenge_scale": C.get("clauseC_literal_revenge_scale",
                                               "not applicable"),
        # The second backward-looking site, for reporting only (design section 0).
        "r_own_Q333": SECOND_SITE_REPORT.get(facet, {}).get("r_own", np.nan),
        "r_own_Q333_lo": SECOND_SITE_REPORT.get(facet, {}).get("lo", np.nan),
        "r_own_Q333_hi": SECOND_SITE_REPORT.get(facet, {}).get("hi", np.nan),
        "top_rival_Q333": SECOND_SITE_REPORT.get(facet, {}).get("top_rival") or "not run",
        "delta_r_Q333": SECOND_SITE_REPORT.get(facet, {}).get("delta", np.nan),
    })
gate_results = pd.DataFrame(gate_rows)
GATE_BACKED_FACETS = gate_results.loc[gate_results["status"] == "GATE_BACKED",
                                      "facet"].tolist()


In [ ]:

# === DO (15 of 15, cont.): the proportionality base rate, for the record ===
# BASE RATE means the share of responses in which the facet appears at all -- how
# often people actually said this kind of thing. It is the number that justifies
# routing proportionality to descriptive-only, and it is computed from this corpus
# rather than assumed from the pilot.
def realized_base_rate(facet, item):
    col = f"ens_{facet}"
    if col not in feat.columns:
        return np.nan
    sub = feat[feat["item_type"] == item][col].dropna()
    # CLEAR presence: the ensemble mean above PRESENT_CLEAR (.50), i.e. the models on
    # balance called the facet present. Fixed in design section 0 before scoring and
    # used only where a facet has to be COUNTED; every correlation and every clause
    # uses the continuous mean. Trace presence (any score above zero) is printed
    # beside it below, so the definition in use is visible on the page.
    return float((sub > PRESENT_CLEAR).mean()) if len(sub) else np.nan

prop_br = {it: realized_base_rate("proportionality", it)
           for it in ["backward", "ranking"]}
print(f"PROPORTIONALITY realized ensemble base rate (clear = mean above "
      f"{PRESENT_CLEAR:.2f}; trace = above zero):")
for it, v in prop_br.items():
    _tr = (feat[feat["item_type"] == it]["ens_proportionality"].dropna()
           if "ens_proportionality" in feat.columns else pd.Series(dtype=float))
    _t = float((_tr > 0).mean()) if len(_tr) else np.nan
    print(f"  {it:10s} clear {'--' if np.isnan(v) else f'{100*v:.1f}%'}   "
          f"trace {'--' if np.isnan(_t) else f'{100*_t:.1f}%'}")
# The 10% floor: below one response in ten there are too few proportionality-positive
# cases for the desert-versus-proportionality dissociation to have anywhere to
# appear. A near-zero correlation drawn from that corpus is an absence of
# measurement, not evidence that the two constructs come apart -- so E5's text-side
# result is reported as a DESCRIPTIVE NULL-BY-ABSENCE rather than as a null.
prop_low = all((not np.isnan(v)) and v < 0.10 for v in prop_br.values())
print(f"  -> E5's text-side desert-vs-proportionality result is reported as a")
print(f"     descriptive null-by-absence: {prop_low}")
gate_results["prop_null_by_absence"] = prop_low


In [ ]:

# === DO (15 of 15, cont.): read the gate table ===
print(f"\n=== GATE TABLE ===   mode: {GATE_MODE}")
_cols = ["facet", "status", "gate_eligible", "clauseA", "clauseB", "clauseC",
         "clauseC_registered_revenge_scale", "clauseC_literal_revenge_scale",
         "icc2k", "icc2k_ci_width", "r_own", "r_own_ci_width", "reason"]
# na_rep, not a fallback inside float_format: pandas renders a null through na_rep
# and never hands it to float_format, so the else branch of such a lambda is dead
# code and the table prints the word NaN. This notebook says an uncomputable
# quantity shows as -- and never as a blank; na_rep is what makes that true.
print(gate_results[_cols].to_string(
    index=False, float_format=lambda x: f"{x:.3f}", na_rep="--"))

print("\nEVERY CLAUSE IS A WORD, NEVER A BLANK: pass, fail, or not run. A clause")
print("that could not be applied is not a clause the facet failed.")
if not HUMAN_GATE_ACTIVE:
    print("  clause A reads 'not run' on every row. It is this design's addition to")
    print("  the registration and it is switched off (HUMAN_GOLD = False), so the")
    print("  gate applied here is the two-clause gate prereg 6.7 actually names.")
print("  censure and moral_balance read 'not run' on clause C because prereg 5.15")
print("  gives them no survey anchor and the instrument has no items for them.")

print(f"\nGate-backed (primary standing inside the exploratory family): "
      f"{GATE_BACKED_FACETS if GATE_BACKED_FACETS else 'none'}")
# HAZARD: this list drives the facet sets of the E3 and E7 notebooks. If it comes
# back empty, those notebooks fall back to the gate-eligible trio and say so in
# their output -- read that warning as "descriptive", never as a passing gate.
if not GATE_BACKED_FACETS:
    print("  NOTHING CLEARED THE GATE. That is a result about the measurement, and")
    print("  it is reported as one: every E3-E7 language finding is then descriptive.")
print("NOTE: gate-backed is not confirmatory. All of E3-E7 is exploratory (2.4).")


## Save and check


In [ ]:

# === SAVE AND CHECK ===
# Three files leave this notebook. Each is checked against its own key contract,
# written to disk, and registered with done() last, so the manifest can never claim
# a file that is not there and never claims a file with a duplicated key.
#   gate_results.csv                  one row per facet: every clause, every estimate,
#                                     every interval and its width, and the reason
#                                     behind every standing.
#   stream_agreement_matrix.csv       facet x item: how far the model ensemble, the
#                                     word list and the embeddings agree.
#   gate_c_facet_by_scale_matrix.csv  every facet against every survey scale, so the
#                                     off-diagonal clause C was decided on can be
#                                     read rather than taken on trust (design 7).
#
# WHY THE CHECKS ARE HERE AND NOT ONLY UPSTREAM. Notebooks 11 and 12 read
# gate_results.csv facet by facet. A facet appearing twice would give one facet two
# standings, and whichever row sorted last would silently become the verdict.
assert not gate_results.duplicated(["facet"]).any(), (
    "gate_results.csv must hold exactly one row per facet")
assert len(gate_results) == len(FACETS), (
    f"gate_results.csv has {len(gate_results)} rows for {len(FACETS)} facets")
gate_results.to_csv("gate_results.csv", index=False)
done("gate_results.csv", rows=len(gate_results),
     note=f"gate mode {GATE_MODE}; clause floors ICC {ICC_FLOOR}, r {R_OWN_FLOOR}, "
          f"decided on the 95% lower bound; {N_BOOT} resamples")

assert not agreement_matrix.duplicated(["facet", "item_type"]).any(), (
    "stream_agreement_matrix.csv must hold one row per facet per item")
agreement_matrix.to_csv("stream_agreement_matrix.csv", index=False)
done("stream_agreement_matrix.csv", rows=len(agreement_matrix),
     note="facet x item_type; inter-model, word-list and embedding legs; "
          "human leg not run unless HUMAN_GOLD")

gate_c_matrix = pd.DataFrame(gate_c_rows) if gate_c_rows else pd.DataFrame(
    columns=["facet", "survey_scale", "is_own", "partial_r", "ci_lo", "ci_hi",
             "ci_width", "n"])
assert not gate_c_matrix.duplicated(["facet", "survey_scale"]).any(), (
    "gate_c_facet_by_scale_matrix.csv must hold one row per facet per scale; a "
    "duplicate would put two different partial correlations in one matrix cell")
gate_c_matrix.to_csv("gate_c_facet_by_scale_matrix.csv", index=False)
done("gate_c_facet_by_scale_matrix.csv", rows=len(gate_c_matrix),
     note="full facet x survey scale partial correlations at the gate item")


In [ ]:

# === SAVE AND CHECK (cont.): what was produced, and what is still not known ===
print(f"\nNOTEBOOK {NB_ID} DONE -- {NB_NAME}")
print(f"  {NB_ASKS}\n")
print("WHAT WAS PRODUCED")
_counts = gate_results["status"].value_counts().to_dict()
for _s in ["GATE_BACKED", "DESCRIPTIVE_ONLY", "CONVERGENT_SANITY"]:
    print(f"  {_s:18s} {_counts.get(_s, 0)} facets")
for _c in ["clauseA", "clauseB", "clauseC"]:
    _v = gate_results[_c].value_counts().to_dict()
    print(f"  {_c}: " + ", ".join(f"{k} = {v}" for k, v in sorted(_v.items())))

print("\nWHAT IS STILL NOT KNOWN")
print("  Clause B says the models agreed with each other, which is reliability and")
print("  not accuracy. Clause C says a facet tracks a survey scale, which is")
print("  convergence with a self-report instrument that may share the facet's own")
print("  blind spots -- the survey's desert and revenge scales correlate about .61,")
print("  so the survey cannot cleanly arbitrate that boundary either.")
if not HUMAN_GATE_ACTIVE:
    print("  Clause A did not run, so nothing in this chain has compared the models")
    print("  against a human reading of the same responses. Prereg 6.7 prescribes no")
    print("  human step, so this run is the registered design exactly; the gap it")
    print("  leaves is the design's, and it belongs in the limitations paragraph.")
print("  Nothing here is confirmatory. Prereg 2.4 puts all of E3-E7 in one")
print("  exploratory family and the gate decides standing inside it, nothing more.")
print("\n  next: the E3-E7 analysis notebooks, which read gate_results.csv to decide")
print("  which facets carry primary weight and which are reported as descriptive.")
